# Valorant VCT Match Predictor

A machine-learning pipeline that predicts pro Valorant matches and shows each team's win probability.

The notebook is built **one layer at a time**:

| Layer | Purpose | Status |
|---|---|---|
| **0. Environment** | Libraries, Google Drive, config, connectivity check | ✅ done |
| **1. Ingestion** | Collect match data from vlr.gg | ✅ done |
| **2. Cleaning** | Team IDs, validation, tidy tables | ✅ done |
| **3. Features** | Elo, roster rating, map pool, form (leak-free) | ✅ done |
| **4. Dataset** | Time-based splits, mirrored rows | ✅ done |
| **5. Models** | Elo baseline, Logistic Regression, LightGBM | ✅ done |
| **6. Maps** | Map model, veto simulation, map-by-map breakdown | ✅ done |
| **7. Calibration** | Trustworthy percentages, confidence notes, `predict_match()` | ✅ done |
| **8. Evaluation** | Final score on the locked test block, with error bars | ✅ done |
| **9. App** | Gradio app: pick two teams, see win %, upcoming matches | ✅ done |
| 10. Operations | Save, refresh, monitor | next |

**How to run:** `Runtime → Run all`, or run cells top to bottom with `Shift + Enter`.
No GPU is needed — leave the runtime on CPU.

---
# Layer 0 — Environment

This layer prepares everything the later layers rely on:

1. **Install libraries** that Colab doesn't already have.
2. **Mount Google Drive**, so downloaded data and trained models survive when Colab disconnects.
3. **Create one config object** — every setting lives here, so you never hunt through code to change something.
4. **Create folders, set random seeds, start a log file.**
5. **Pre-flight check** — test whether Colab can actually reach vlr.gg. This decides how Layer 1 collects data.
6. **Environment summary** — confirms everything is ready.

### 0.1 Install libraries
Takes about 1–2 minutes the first time in each Colab session. Colab already has most of these; pip skips what's present.

In [ ]:
import os, sys, subprocess

PACKAGES = [
    "pandas", "numpy", "pyarrow",        # data tables + fast file format
    "requests", "httpx", "tenacity",     # web requests + automatic retries
    "selectolax==0.4.12", "cachetools",  # HTML parsing for the vlr.gg parsers (Layer 1). Pinned:
                                         # selectolax 1.0 removed a module the parsers import.
    "fastapi",                           # only imported by the vlr.gg parser library
    "scikit-learn", "lightgbm",          # models (Layer 5)
    "joblib",                            # saving models
    "shap",                              # explaining predictions (Layer 8)
    "gradio",                            # the web app (Layer 9)
]

# Set to True to skip installing (e.g. you already ran this cell in this session).
SKIP_INSTALL = os.environ.get("VALPRED_SKIP_INSTALL") == "1"

if SKIP_INSTALL:
    print("Skipping install.")
else:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("pip install failed — read the error above.")
    print("✅ Libraries installed.")

### 0.2 Mount Google Drive
Colab will ask permission to access your Drive the first time — allow it.
Everything is stored in **`MyDrive/valorant_predictor/`**. If you run this notebook outside Colab, a local folder is used instead.

In [ ]:
import os, importlib.util
from pathlib import Path

IS_COLAB = importlib.util.find_spec("google.colab") is not None

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/valorant_predictor")
else:
    PROJECT_DIR = Path(os.environ.get("VALPRED_PROJECT_DIR", Path.cwd() / "valorant_predictor"))

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Running in Colab: {IS_COLAB}")
print(f"Project folder:   {PROJECT_DIR}")

### 0.3 Configuration
**Every setting for the whole project lives here.** Change a value, re-run this cell, and later layers pick it up.

Key choices and why:
- `start_date = 2023-01-01` — older matches come from a different game meta and different rosters, so they add noise.
- `request_delay_s = 1.5` — wait between requests so we don't hammer vlr.gg (and don't get blocked).
- The **four time blocks** (train → tune → calibrate → test) always follow date order. The test block is the newest matches and is opened only once, at the very end — that's what keeps the final score honest.

In [ ]:
import json, datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict, field

@dataclass
class Config:
    project_dir: str

    # --- Data scope ---
    start_date: str = "2023-01-01"
    event_tiers: tuple = ("international", "regional", "challengers")

    # --- Data collection (Layer 1) ---
    vlr_base: str = "https://www.vlr.gg"
    # HTML parsers come from the open-source vlrggapi project, pinned to one tested commit
    # so a surprise update can't break collection. Update deliberately if vlr.gg changes layout.
    api_repo: str = "https://github.com/axsddlr/vlrggapi"
    api_commit: str = "0af635b22be80d4a5d67c3b212ebafcba145a3ac"
    # Which events count is decided by EVENT_RULES in cell 0.3b (exact official-name patterns).
    max_event_pages: int = 120                 # safety cap when paging back through events
    # Which tiers get full map/player detail (Stage B, ~1 request per match).
    # Add "challengers" for more data — roughly triples Stage B time.
    detail_tiers: tuple = ("international", "regional")
    stage_b_max_minutes: float = 60.0          # stop Stage B after this long; re-run to continue
    refresh_ongoing_hours: float = 3.0         # re-check events still in progress after this long
    save_html: bool = True                     # keep raw pages so we can re-parse without re-downloading
    request_delay_s: float = 1.5
    max_retries: int = 4
    request_timeout_s: float = 20.0
    chunk_size: int = 25                       # save progress every N matches
    user_agent: str = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")

    # --- Time-ordered split (fractions of matches, oldest → newest) ---
    split_train: float = 0.65
    split_tune: float = 0.12
    split_calibrate: float = 0.08
    split_test: float = 0.15

    # --- Modelling ---
    seed: int = 42
    min_maps_for_confidence: int = 20          # fewer maps than this → "low confidence" flag
    vct_weight: float = 3.0                    # a VCT match counts this many times in training and model choice
    calibration_prior: float = 5.0             # how strongly calibration is held near "no change" (small data → cautious)
    bootstrap_samples: int = 1000              # for error bars (Layer 8)
    veto_simulations: int = 2000               # for series probability (Layer 6)

    # --- Folder names inside project_dir ---
    folders: dict = field(default_factory=lambda: {
        "raw": "data/raw",            # untouched API responses (JSON)
        "clean": "data/clean",        # tidy tables (Parquet)
        "features": "data/features",  # model-ready features
        "models": "models",           # trained models + calibrators
        "logs": "logs",               # run logs, pre-flight results, prediction log
        "src": "src",                 # helper code (later layers)
        "events": "data/raw/events",              # event list (Layer 1)
        "event_matches": "data/raw/event_matches",  # one file per event (Layer 1, Stage A)
        "match_details": "data/raw/match_details",  # map + player detail (Layer 1, Stage B)
    })

    def path(self, key: str) -> Path:
        return Path(self.project_dir) / self.folders[key]

    def validate(self):
        total = self.split_train + self.split_tune + self.split_calibrate + self.split_test
        assert abs(total - 1.0) < 1e-9, f"Split fractions must sum to 1, got {total}"
        assert all(0 < x < 1 for x in (self.split_train, self.split_tune,
                                        self.split_calibrate, self.split_test))
        dt.date.fromisoformat(self.start_date)          # raises if the date is malformed
        assert self.request_delay_s >= 0.5, "Keep at least 0.5 s between requests"
        assert self.max_retries >= 1 and self.chunk_size >= 1
        assert set(self.detail_tiers) <= {"international", "regional", "challengers"}, \
            "detail_tiers can only contain: international, regional, challengers"
        return self

CFG = Config(project_dir=str(PROJECT_DIR)).validate()

config_file = PROJECT_DIR / "config.json"
config_file.write_text(json.dumps(asdict(CFG), indent=2))
print(f"✅ Config valid — saved to {config_file}")
print(f"   Data from {CFG.start_date} | split train/tune/cal/test = "
      f"{CFG.split_train}/{CFG.split_tune}/{CFG.split_calibrate}/{CFG.split_test}")

### 0.3b Which events count
Only **official Riot events** are used. Official names follow fixed patterns, so we match whole patterns rather than single words — matching single words let in look-alikes such as *"College VALORANT **Champions**hip"* or third-party *"… Masters"* cups, which put college teams into the VCT ratings.

| Tier | Official name patterns (examples) |
|---|---|
| `international` | *Valorant Champions 2025* · *Valorant Masters Toronto 2025* · *Champions Tour 2024: Masters Madrid* · *Champions Tour 2023: LOCK//IN São Paulo* |
| `regional` | *VCT 2026: EMEA Stage 1* · *VCT 2025: Pacific Kickoff* · *Champions Tour 2023: Americas League* |
| `challengers` | *Challengers 2025: North America ACE Stage 1* · *Champions Tour 2023: Pacific Ascension* |
| left out | Game Changers, college/university, showmatches, anything else |

Rules are checked top to bottom; the first match decides. Layer 2 re-applies them to the saved data, so changing a rule never needs a re-download.

In [ ]:
import re

EVENT_RULES = [
    ("excluded",      r"game changers|college|collegiate|university|showmatch|academy|\bnecc\b"),
    ("international", r"^valorant champions 20\d\d\b|^valorant masters\b|^champions tour 20\d\d: (masters|lock//in)\b"),
    ("challengers",   r"^(valorant |vct )?challengers\b|^(vct|champions tour) 20\d\d: .*\b(challengers|ascension)\b"),
    ("regional",      r"^(vct|champions tour)\b.*\b(americas|emea|pacific|china)\b"),
]

def classify_event(title: str):
    """'international' / 'regional' / 'challengers' for official events, else None."""
    t = re.sub(r"\s+", " ", (title or "").strip().lower())
    for tier, pattern in EVENT_RULES:
        if re.search(pattern, t):
            return None if tier == "excluded" else tier
    return None

def event_region(title: str):
    """The VCT region an event belongs to (from its name), or None."""
    m = re.search(r"\b(americas|emea|pacific|china)\b", (title or "").lower())
    return m.group(1) if m else None

for title in ["Valorant Champions 2025", "Valorant Masters Toronto 2025", "Champions Tour 2024: Masters Madrid",
              "VCT 2026: EMEA Stage 1", "Challengers 2025: North America ACE Stage 1",
              "College VALORANT Championship 2025", "NECC Valorant Champions Finals - Spring 2023",
              "FunPay Clutch Masters", "Game Changers 2025: EMEA"]:
    print(f"   {title:46s} → {classify_event(title)}")

### 0.4 Folders, random seeds, and logging
- **Folders:** one place for each kind of file (see `folders` in the config).
- **Seeds:** fixing the random seed makes results repeatable — run twice, get the same numbers.
- **Logging:** messages go to the screen *and* to `logs/run.log`, so you can see what happened even after Colab disconnects.

In [ ]:
import os, sys, random, logging
import numpy as np

for key in CFG.folders:
    CFG.path(key).mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
os.environ["PYTHONHASHSEED"] = str(CFG.seed)

def get_logger(name: str = "valpred") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:                      # don't add duplicate handlers on re-run
        return logger
    logger.setLevel(logging.INFO)
    logger.propagate = False                 # stop Colab's own logger printing every line twice
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    file_handler = logging.FileHandler(CFG.path("logs") / "run.log", encoding="utf-8")
    file_handler.setFormatter(fmt)
    screen_handler = logging.StreamHandler(sys.stdout)
    screen_handler.setFormatter(fmt)
    logger.addHandler(file_handler)
    logger.addHandler(screen_handler)
    return logger

log = get_logger()
log.info("Layer 0 started — folders ready, seed=%d", CFG.seed)
for key in CFG.folders:
    print(f"  {key:13s} → {CFG.path(key)}")

### 0.5 Pre-flight connectivity check
vlr.gg is protected by Cloudflare, which **sometimes blocks requests from Google Cloud servers — and Colab runs on Google Cloud**. Instead of finding out halfway through a 2-hour download, we test right now.

Each site gets one of these results:

| Result | Meaning |
|---|---|
| `ok` | Reachable and the page looks as expected |
| `blocked` | Cloudflare (or a similar shield) refused us |
| `unexpected_layout` | Reachable, but the page structure changed — the scraper may need an update |
| `http_XXX` | Some other HTTP error |
| `network_error` | Couldn't connect at all (timeout, DNS, no internet) |

The first cell defines the checker; the second runs it.

In [ ]:
import requests

def check_url(url: str, expect: str | None = None, timeout: float | None = None) -> dict:
    """Request one URL and classify the outcome (see the table above)."""
    timeout = timeout or CFG.request_timeout_s
    try:
        r = requests.get(url, headers={"User-Agent": CFG.user_agent}, timeout=timeout)
    except requests.RequestException as e:
        return {"url": url, "status": "network_error", "http": None,
                "detail": f"{type(e).__name__}: {str(e)[:150]}"}

    header_names = {k.lower() for k in r.headers}
    behind_cloudflare = ("cloudflare" in r.headers.get("server", "").lower()
                         or "cf-ray" in header_names)
    body = r.text[:20000].lower()
    challenge_page = any(s in body for s in ("just a moment", "cf-chl", "attention required"))

    if r.status_code in (403, 429, 503) and (behind_cloudflare or challenge_page):
        status = "blocked"
    elif r.status_code != 200:
        status = f"http_{r.status_code}"
    elif challenge_page:
        status = "blocked"                     # 200 but it's a challenge page, not real content
    elif expect and expect.lower() not in body:
        status = "unexpected_layout"
    else:
        status = "ok"
    return {"url": url, "status": status, "http": r.status_code,
            "seconds": round(r.elapsed.total_seconds(), 2)}


def choose_plan(checks: dict) -> tuple[str, str]:
    """Turn the check results into a data-collection plan for Layer 1."""
    vlr = checks["vlr_results"]["status"]
    github = checks["github"]["status"]
    if vlr == "ok" and github == "ok":
        return "A", ("Plan A: collect data inside Colab. Layer 1 downloads the vlr.gg "
                     "parsers from GitHub and runs the collection here.")
    if vlr == "ok":
        return "A-direct", ("Plan A-direct: vlr.gg works but GitHub doesn't, so Layer 1 can't download "
                            "the parser library. Download it as a ZIP on your laptop from "
                            f"{CFG.api_repo} and upload it to MyDrive/valorant_predictor/src/vlrggapi.zip "
                            "— Layer 1 will use that copy.")
    if vlr == "unexpected_layout":
        return "A-check", ("vlr.gg is reachable but its results page looks different from expected. "
                           "Layer 1 will try anyway and stop early with a warning if pages can't be read.")
    return "B", ("Plan B: Colab can't reach vlr.gg right now. Run the Layer 1 collector on your "
                 "laptop (instructions will be in Layer 1), upload the files to "
                 "MyDrive/valorant_predictor/data/raw/, and continue in Colab from Layer 2. "
                 "Tip: first try Runtime → Disconnect and delete runtime, then re-run — "
                 "a new Colab machine sometimes gets a different IP that isn't blocked.")

In [ ]:
import json, datetime as dt

CHECKS = {
    "vlr_home":    check_url("https://www.vlr.gg/"),
    "vlr_results": check_url("https://www.vlr.gg/matches/results", expect="match-item"),
    "github":      check_url(CFG.api_repo),
    "pypi":        check_url("https://pypi.org/simple/lightgbm/"),
}

icons = {"ok": "✅", "blocked": "⛔", "network_error": "🔌", "unexpected_layout": "⚠️"}
print(f"{'check':12s} {'result':18s} {'http':5s} url")
for name, c in CHECKS.items():
    icon = icons.get(c["status"], "❌")
    print(f"{name:12s} {icon} {c['status']:15s} {str(c['http']):5s} {c['url']}")
    if "detail" in c:
        print(f"{'':12s}    ↳ {c['detail']}")

PLAN, PLAN_TEXT = choose_plan(CHECKS)
print(f"\n➡️  {PLAN_TEXT}")

preflight = {"time_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
             "is_colab": IS_COLAB, "plan": PLAN, "checks": CHECKS}
(CFG.path("logs") / "preflight.json").write_text(json.dumps(preflight, indent=2))
log.info("Pre-flight done: plan=%s", PLAN)

### 0.6 Environment summary
A final checklist. If every line is ✅, Layer 0 is done and you can move on to Layer 1.

In [ ]:
import sys, json, platform, shutil
from importlib.metadata import version, PackageNotFoundError

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

def ram_gb():
    try:
        with open("/proc/meminfo") as f:
            kb = int(next(l for l in f if l.startswith("MemTotal")).split()[1])
        return round(kb / 1024**2, 1)
    except Exception:
        return None

required = ["pandas", "numpy", "pyarrow", "requests", "httpx", "tenacity", "selectolax",
            "cachetools", "fastapi", "scikit-learn", "lightgbm", "joblib", "shap", "gradio"]
versions = {p: pkg_version(p) for p in required}
missing = [p for p, v in versions.items() if v is None]

disk_free_gb = round(shutil.disk_usage(PROJECT_DIR).free / 1024**3, 1)
summary = {
    "python": platform.python_version(),
    "ram_gb": ram_gb(),
    "disk_free_gb": disk_free_gb,
    "packages": versions,
    "plan": PLAN,
}

def line(ok, text):
    print(("✅ " if ok else "❌ ") + text)

line(sys.version_info >= (3, 10), f"Python {summary['python']} (need 3.10+)")
line(not missing, "All libraries installed" if not missing else f"Missing libraries: {missing} — re-run 0.1")
line(disk_free_gb > 1, f"Free space where data is saved: {disk_free_gb} GB (need > 1 GB)")
line(PROJECT_DIR.exists() and config_file.exists(), "Project folder and config.json exist")
line(PLAN in ("A", "A-direct", "A-check", "B"), f"Data plan chosen: {PLAN}")
if summary["ram_gb"]:
    print(f"ℹ️  RAM: {summary['ram_gb']} GB (plenty — this project needs < 2 GB)")

(CFG.path("logs") / "layer0_status.json").write_text(json.dumps(summary, indent=2))
LAYER0_OK = not missing and disk_free_gb > 1
log.info("Layer 0 finished: ok=%s plan=%s", LAYER0_OK, PLAN)
print("\n🎉 Layer 0 complete." if LAYER0_OK else "\nFix the ❌ items above, then re-run this cell.")

---
# Layer 1 — Data Ingestion

Collects pro Valorant match data from **vlr.gg** and saves it to your Drive.

**How it works**
- vlr.gg has no official API, so we read its web pages. The page-reading code (the *parsers*) comes from the open-source **vlrggapi** project, pinned to one tested version.
- We fetch pages **politely**: one at a time, ~1.5 s apart, with automatic retries. Being polite is also what keeps us from getting blocked.

**Three steps**

| Step | What | Requests | Time |
|---|---|---|---|
| **1.3 Find events** | Page through vlr.gg's completed events, keep VCT + Challengers since 2023 | ~1 per page | a few minutes |
| **1.4 Stage A — results** | For each event, the list of its matches: teams, score, date | 1 per event | ~10–20 min |
| **1.5 Stage B — details** | For each tier-1 match: maps, round scores, all 10 players' stats | 1 per match | **1–3 hours total** |

**Everything resumes.** Each step skips work already saved on Drive. If Colab disconnects, just run the cells again — it continues where it stopped.

### 1.1 Get the vlr.gg parser library
Downloads vlrggapi at the exact commit set in `CFG.api_commit` and checks that it imports and parses correctly.
We use only its page-reading functions, not its web server — that lets us fetch **one page per match** instead of up to seven.

In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path
from importlib.metadata import version as _pkg_version

if not _pkg_version("selectolax").startswith("0.4."):
    raise RuntimeError(f"selectolax {_pkg_version('selectolax')} is installed, but the parsers need 0.4.12. "
                       "Re-run cell 0.1, then Runtime → Restart session, then run all cells again.")

LIB_DIR = (Path("/content/vlrggapi") if IS_COLAB
           else Path(os.environ.get("VALPRED_LIB_DIR", PROJECT_DIR.parent / "vlrggapi_lib")))

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr[-1500:]}")
    return r.stdout.strip()

def get_parser_library() -> str:
    marker = LIB_DIR / ".valpred_commit"
    if marker.exists() and marker.read_text().strip() == CFG.api_commit:
        return f"already present (commit {CFG.api_commit[:7]})"
    shutil.rmtree(LIB_DIR, ignore_errors=True)
    LIB_DIR.mkdir(parents=True)
    try:
        _run(["git", "init", "-q"], cwd=LIB_DIR)
        _run(["git", "remote", "add", "origin", CFG.api_repo], cwd=LIB_DIR)
        _run(["git", "fetch", "-q", "--depth", "1", "origin", CFG.api_commit], cwd=LIB_DIR)
        _run(["git", "checkout", "-q", "FETCH_HEAD"], cwd=LIB_DIR)
        marker.write_text(CFG.api_commit)
        return f"downloaded from GitHub (commit {CFG.api_commit[:7]})"
    except Exception as git_error:
        zip_copy = CFG.path("src") / "vlrggapi.zip"
        if not zip_copy.exists():
            raise RuntimeError(f"Couldn't download the parser library from GitHub and there is no "
                               f"uploaded copy at {zip_copy}.\n{git_error}")
        shutil.rmtree(LIB_DIR, ignore_errors=True)
        unpack = LIB_DIR.parent / "_vlrggapi_zip"
        shutil.rmtree(unpack, ignore_errors=True)
        with zipfile.ZipFile(zip_copy) as z:
            z.extractall(unpack)
        top = next(p for p in unpack.iterdir() if p.is_dir())   # GitHub ZIPs have one top folder
        shutil.move(str(top), LIB_DIR)
        (LIB_DIR / ".valpred_commit").write_text("uploaded-zip")
        return "loaded from your uploaded ZIP (version not pinned)"

print("Parser library:", get_parser_library())

# The library uses generic package names (api, utils, models), so make sure Python
# loads *this* copy and not some other package with the same name.
for name in list(sys.modules):
    if name.split(".")[0] in {"api", "utils", "models", "routers"}:
        del sys.modules[name]
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import utils as _vlr_utils
assert Path(_vlr_utils.__file__).resolve().parent.parent == LIB_DIR.resolve(), \
    f"Wrong 'utils' package loaded: {_vlr_utils.__file__}"

from fastapi import HTTPException
from utils.html_parsers import parse_html
from utils.http_client import fetch_with_retries, CircuitOpenError
import api.scrapers.events as vlr_events_mod
from api.scrapers.events import _parse_event_cards
from api.scrapers.match_detail.parsers import (
    _parse_event_info, _parse_match_header, _parse_teams, _parse_maps, _parse_head_to_head)

# Point the library at the configured site (normally https://www.vlr.gg).
VLR = CFG.vlr_base.rstrip("/")
vlr_events_mod.VLR_BASE_URL = VLR
vlr_events_mod.VLR_EVENTS_URL = f"{VLR}/events"

# Smoke test: parse a tiny hand-made match page.
_sample = parse_html("""
  <a class="match-header-link mod-1" href="/team/2/sentinels"></a>
  <div class="match-header-link-name mod-1">
    <div>Sentinels</div>
    <div>SEN</div>
  </div>
  <a class="match-header-link mod-2" href="/team/188/cloud9"></a>
  <div class="match-header-link-name mod-2">
    <div>Cloud9</div>
    <div>C9</div>
  </div>""")
_t = _parse_teams(_sample)
assert [(t["id"], t["name"], t["tag"]) for t in _t] == [("2", "Sentinels", "SEN"), ("188", "Cloud9", "C9")], _t
print("✅ Parsers imported and working.")

### 1.2 Helper functions
- **`classify_event`** (defined in 0.3b) — decides if an event is official, and its tier.
- **`polite_get`** — waits ~1.5 s (with a little randomness) before each request and retries failures. If vlr.gg is slow or a connection drops, it reports a `NetworkError` so the caller can skip that page and come back to it later. If vlr.gg starts **blocking** us, it stops immediately instead of hammering it.
- **`save_json`** — writes to a temporary file first, then renames. A disconnect mid-write can never leave a half-written, corrupt file.

In [ ]:
import re, json, gzip, time, random, asyncio, datetime as dt
import httpx
from dateutil import parser as date_parser

class BlockedError(RuntimeError):
    """vlr.gg is refusing our requests — stop instead of retrying."""

class NetworkError(RuntimeError):
    """vlr.gg didn't answer in time, or the connection dropped (already retried)."""

def title_year(title: str):
    m = re.search(r"\b(20[2-3]\d)\b", title or "")
    return int(m.group(1)) if m else None

def parse_match_date(date_text: str, unix_ts: str = ""):
    """Event pages give dates like 'Wed, September 30, 2026'. The date text is trusted first:
    the parser library's timestamp can be a year off for older matches."""
    if date_text and re.search(r"\b20\d\d\b", date_text):
        try:
            return date_parser.parse(date_text, fuzzy=True).date()
        except (ValueError, OverflowError):
            pass
    if unix_ts:
        try:
            return dt.datetime.strptime(unix_ts[:10], "%Y-%m-%d").date()
        except ValueError:
            pass
    return None

def _looks_blocked(resp) -> bool:
    body = resp.text[:5000].lower() if resp.text else ""
    return (resp.status_code in (403, 429, 503) and
            ("cloudflare" in resp.headers.get("server", "").lower() or "just a moment" in body)) \
        or "cf-chl" in body

async def polite_get(url: str):
    await asyncio.sleep(CFG.request_delay_s * random.uniform(0.8, 1.3))
    for _ in range(3):
        try:
            resp = await fetch_with_retries(url, max_retries=CFG.max_retries,
                                            request_delay=CFG.request_delay_s,
                                            timeout=CFG.request_timeout_s)
        except CircuitOpenError:
            log.warning("vlr.gg keeps failing — pausing 60 s before trying again")
            await asyncio.sleep(60)
            continue
        except httpx.HTTPError as e:     # timeout, dropped connection, DNS hiccup — after all retries
            raise NetworkError(f"no answer from vlr.gg after {CFG.max_retries} tries "
                               f"({type(e).__name__})") from None
        if _looks_blocked(resp):
            raise BlockedError(f"vlr.gg blocked the request to {url} (HTTP {resp.status_code}). "
                               "Wait ~30 minutes, or use Runtime → Disconnect and delete runtime "
                               "to get a new Colab machine, then re-run. Saved progress is kept.")
        return resp
    raise RuntimeError(f"vlr.gg kept failing for {url}; try again later (progress is saved).")

def save_json(path: Path, obj) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, default=str))
    tmp.replace(path)

def load_json(path: Path, default=None):
    try:
        return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return default

START_DATE = dt.date.fromisoformat(CFG.start_date)
print("✅ Helpers ready.")

### 1.3 Find events
Keeps the official events (rules in cell 0.3b):
- **Events in progress** (e.g. Champions while it's being played) from the top of vlr.gg's events page — these hold the most recent matches, which matter most for predicting what comes next.
- **Completed events**, paging backwards until two pages in a row contain only events from before `CFG.start_date`'s year.

The list is saved and reused for a day; set `REFRESH_EVENTS = True` to force a fresh check.

In [ ]:
REFRESH_EVENTS = False

EVENTS_FILE = CFG.path("events") / "events.json"

async def discover_events() -> dict:
    events, old_pages, page = {}, 0, 0
    for page in range(1, CFG.max_event_pages + 1):
        url = f"{VLR}/events" if page == 1 else f"{VLR}/events/?page={page}"
        resp = await polite_get(url)
        if resp.status_code == 404:
            break
        if resp.status_code >= 400:
            raise RuntimeError(f"Events page {page} returned HTTP {resp.status_code}")
        html = parse_html(resp.text)
        if page == 1:                                        # events in progress are listed first
            for section in html.css("div.wf-label.mod-large.mod-upcoming"):
                for c in (_parse_event_cards(section.parent) if section.parent else []):
                    tier = classify_event(c["title"])
                    if tier and c["event_id"] and c["status"].strip().lower() == "ongoing":
                        events[c["event_id"]] = {**c, "tier": tier, "year": title_year(c["title"]),
                                                 "list_page": 0, "state": "ongoing"}
        cards = []
        for section in html.css("div.wf-label.mod-large.mod-completed"):
            if section.parent:
                cards.extend(_parse_event_cards(section.parent))
        if not cards:
            if page == 1:
                raise RuntimeError("No events found on vlr.gg's events page — its layout may have "
                                   "changed. Check for a newer vlrggapi commit and update CFG.api_commit.")
            break                                            # ran past the last page
        years = [y for y in (title_year(c["title"]) for c in cards) if y]
        for c in cards:
            tier, year = classify_event(c["title"]), title_year(c["title"])
            if tier and c["event_id"] and (year is None or year >= START_DATE.year):
                events[c["event_id"]] = {**c, "tier": tier, "year": year, "list_page": page,
                                         "state": "completed"}
        old_pages = old_pages + 1 if (years and max(years) < START_DATE.year) else 0
        if page % 5 == 0:
            print(f"   page {page}: {len(events)} events kept so far")
        if old_pages >= 2:
            break
    return {"version": 3, "fetched_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "pages_read": page, "events": events}

cached = load_json(EVENTS_FILE)
age_h = None
if cached:
    age_h = (dt.datetime.now(dt.timezone.utc)
             - dt.datetime.fromisoformat(cached["fetched_at"])).total_seconds() / 3600
if cached and cached.get("version") == 3 and not REFRESH_EVENTS and age_h < 24:
    EVENTS = cached["events"]
    print(f"Using saved event list ({age_h:.1f} h old).")
else:
    print("Reading vlr.gg events list…")
    result = await discover_events()
    save_json(EVENTS_FILE, result)
    EVENTS = result["events"]
    log.info("Events found: %d (read %d pages)", len(EVENTS), result["pages_read"])

by_tier = {}
for e in EVENTS.values():
    by_tier[e["tier"]] = by_tier.get(e["tier"], 0) + 1
ongoing = [e["title"] for e in EVENTS.values() if e.get("state") == "ongoing"]
print(f"✅ {len(EVENTS)} events kept: {by_tier}")
print(f"   in progress right now ({len(ongoing)}): {', '.join(ongoing[:6]) or 'none'}")

### 1.4 Stage A — match results for every event
One request per event. Each event's match list is saved as its own file:
- **finished events** are downloaded once and never again;
- **events in progress** are re-checked when their saved copy is older than `CFG.refresh_ongoing_hours` (3 h), and one final time after they finish.
At the end, all results are combined into one table, **`series_index.parquet`** — every completed series with both teams, the score, the date, and the event tier.

In [ ]:
import pandas as pd

async def fetch_event_matches(event_id: str):
    await asyncio.sleep(CFG.request_delay_s * random.uniform(0.8, 1.3))
    for attempt in range(2):
        try:
            result = await vlr_events_mod.vlr_event_matches(event_id)
        except HTTPException as e:
            if e.status_code == 503 and attempt == 0:      # vlr.gg struggling: pause, retry once
                await asyncio.sleep(60)
                continue
            return None, f"HTTP {e.status_code}: {e.detail}"
        data = result.get("data", {})
        if data.get("status", 200) >= 400:
            return None, data.get("error", "upstream error")
        return data.get("segments", []), None
    return None, "vlr.gg unavailable"

def needs_fetch(eid: str, ev: dict) -> bool:
    saved = load_json(CFG.path("event_matches") / f"{eid}.json")
    if not saved:
        return True
    if saved.get("event_state", "completed") == "completed":
        return False                                   # finished when saved → final
    if ev.get("state") == "completed":
        return True                                    # finished since we last looked: fetch final version
    age_h = (dt.datetime.now(dt.timezone.utc)
             - dt.datetime.fromisoformat(saved["fetched_at"])).total_seconds() / 3600
    return age_h >= CFG.refresh_ongoing_hours

todo = [eid for eid, ev in EVENTS.items() if needs_fetch(eid, ev)]
n_refresh = sum(1 for eid in todo if (CFG.path("event_matches") / f"{eid}.json").exists())
print(f"{len(EVENTS) - len(todo)} events up to date, {len(todo)} to fetch "
      f"({n_refresh} of them refreshes of events in progress; "
      f"≈ {len(todo) * (CFG.request_delay_s + 1) / 60:.0f} min).")
failures = {}
t0 = time.time()
for i, eid in enumerate(todo, 1):
    matches, err = await fetch_event_matches(eid)
    if err:
        failures[eid] = err
        continue
    save_json(CFG.path("event_matches") / f"{eid}.json",
              {"event_id": eid, "fetched_at": dt.datetime.now(dt.timezone.utc).isoformat(),
               "event_state": EVENTS[eid].get("state", "completed"), "matches": matches})
    if i % 20 == 0:
        print(f"   {i}/{len(todo)} events  ({(time.time() - t0) / 60:.1f} min)")
if failures:
    print(f"⚠️ {len(failures)} events failed (they'll be retried next run): {list(failures)[:5]}")
    log.warning("Stage A failures: %s", failures)

# Combine every saved event into one table of completed series.
rows = []
for eid, ev in EVENTS.items():
    saved = load_json(CFG.path("event_matches") / f"{eid}.json")
    if not saved:
        continue
    for m in saved["matches"]:
        t1, t2 = m["team1"], m["team2"]
        s1, s2 = str(t1.get("score", "")), str(t2.get("score", ""))
        if (m.get("status", "").strip().lower() != "completed"
                or not (s1.isdigit() and s2.isdigit()) or "TBD" in (t1["name"], t2["name"])):
            continue          # live or not played yet — a live match's details would be half-finished
        match_date = parse_match_date(m.get("date", ""), m.get("unix_timestamp", ""))
        if match_date is None or match_date < START_DATE:
            continue
        rows.append({
            "match_id": m["match_id"], "date": match_date, "event_id": eid,
            "event": ev["title"], "tier": ev["tier"], "stage": m.get("stage", ""),
            "series": m.get("event_series", ""), "note": m.get("note", ""),
            "team1": t1["name"], "team2": t2["name"], "score1": int(s1), "score2": int(s2),
            "winner": 1 if t1.get("is_winner") else (2 if t2.get("is_winner") else 0),
        })

SERIES = (pd.DataFrame(rows).drop_duplicates("match_id")
          .sort_values(["date", "match_id"]).reset_index(drop=True))
if SERIES.empty:
    raise RuntimeError("No completed matches found — check the event list in 1.3 and any errors above.")
SERIES.to_parquet(CFG.path("raw") / "series_index.parquet", index=False)
log.info("Stage A done: %d completed series", len(SERIES))

print(f"\n✅ {len(SERIES)} completed series from {SERIES.date.min()} to {SERIES.date.max()}")
print(pd.crosstab(SERIES.tier, pd.to_datetime(SERIES.date).dt.year, margins=True))

### 1.5 Stage B — map and player details
Downloads one page per match for the tiers in `CFG.detail_tiers` and extracts every map's score (including attack/defence halves), the 10 players' stats, the map veto and head-to-head history.

**This is the long step.** It runs for at most `CFG.stage_b_max_minutes` (60 min), saving every `CFG.chunk_size` matches. To continue, **just run this cell again** — finished matches are skipped. Newest matches are fetched first, so even a partial run gives useful recent data.

- **Stopping early:** prefer the time limit. If you press ■ stop, at most the last unsaved chunk (~1 minute of work) is lost.
- **Safety check:** if most of the first 20 pages can't be read properly, the cell stops instead of saving junk. That usually means vlr.gg changed its layout.

In [ ]:
DETAIL_DIR = CFG.path("match_details")
DONE_FILE = DETAIL_DIR / "done_ids.txt"
FAILED_FILE = DETAIL_DIR / "failed.json"
MAX_ATTEMPTS = 3          # pages that return an error (404, 500…) are given up after 3 runs
MAX_NET_ATTEMPTS = 5      # pages that time out get more chances — slowness is usually temporary
NET_STOP_AFTER = 5        # this many timeouts IN A ROW → vlr.gg is struggling, stop and save

def gave_up(entry: dict) -> bool:
    return entry.get("attempts", 0) >= MAX_ATTEMPTS or entry.get("net_attempts", 0) >= MAX_NET_ATTEMPTS

def load_done() -> set:
    return set(DONE_FILE.read_text().split()) if DONE_FILE.exists() else set()

def record_is_healthy(rec: dict) -> bool:
    teams_ok = all(t.get("id") and t.get("name") for t in rec["teams"])
    maps_ok = any(isinstance(m["score"].get("team1"), int) for m in rec["maps"])
    return teams_ok and maps_ok

async def fetch_match_detail(match_id: str):
    try:
        resp = await polite_get(f"{VLR}/{match_id}")
    except NetworkError as e:
        return None, f"network: {e}"
    if resp.status_code >= 400:
        return None, f"HTTP {resp.status_code}"
    html = parse_html(resp.text)
    rec = {
        "match_id": match_id,
        "fetched_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "event": _parse_event_info(html),
        "header": _parse_match_header(html),       # date, map veto text, status
        "teams": _parse_teams(html),               # vlr.gg team IDs + names + series score
        "maps": _parse_maps(html),                 # per-map scores, halves, players, rounds
        "head_to_head": _parse_head_to_head(html),
    }
    if CFG.save_html:
        rec["html"] = resp.text
    return rec, None

def write_chunk(records: list) -> None:
    """Save a chunk as one compressed file, THEN mark its IDs done (that order is what makes resume safe)."""
    if not records:
        return
    existing = [int(p.name[5:10]) for p in DETAIL_DIR.glob("part-*.jsonl.gz")]
    n = max(existing, default=0) + 1
    final = DETAIL_DIR / f"part-{n:05d}.jsonl.gz"
    tmp = final.with_name(final.name + ".tmp")
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(final)
    with open(DONE_FILE, "a") as f:
        f.write("\n".join(r["match_id"] for r in records) + "\n")

async def run_stage_b() -> dict:
    wanted = SERIES[SERIES.tier.isin(CFG.detail_tiers)].sort_values("date", ascending=False)
    done, failed = load_done(), load_json(FAILED_FILE, {})
    todo = [m for m in wanted.match_id if m not in done and not gave_up(failed.get(m, {}))]
    per_match_s = CFG.request_delay_s * 1.05 + 1.0
    print(f"Detail tiers {CFG.detail_tiers}: {len(wanted)} matches — {len(wanted) - len(todo)} done, "
          f"{len(todo)} to go (≈ {len(todo) * per_match_s / 3600:.1f} h total; "
          f"this run stops after {CFG.stage_b_max_minutes:.0f} min).")

    LAYOUT_SAMPLE = 20
    layout_msg = ("Most match pages couldn't be read correctly (missing team IDs or map scores). "
                  "vlr.gg may have changed its layout — look for a newer vlrggapi commit and update "
                  "CFG.api_commit. Nothing from this run was saved.")
    layout_ok = lambda flags: sum(flags) * 2 >= len(flags)

    buffer, fetched, health, t0 = [], 0, [], time.time()
    net_in_a_row = 0
    layout_checked = False
    stop_reason = "finished"
    try:
        for match_id in todo:
            if (time.time() - t0) / 60 >= CFG.stage_b_max_minutes:
                stop_reason = "time limit"
                break
            rec, err = await fetch_match_detail(match_id)
            if err:
                f = failed.setdefault(match_id, {})
                is_net = err.startswith("network")
                key = "net_attempts" if is_net else "attempts"
                f[key] = f.get(key, 0) + 1
                f["error"] = err
                net_in_a_row = net_in_a_row + 1 if is_net else 0
                if is_net:
                    print(f"   ⚠️ match {match_id}: {err} — skipping for now, will retry next run")
                if net_in_a_row >= NET_STOP_AFTER:
                    stop_reason = "vlr.gg not responding"
                    print(f"⛔ {NET_STOP_AFTER} pages in a row got no answer — vlr.gg seems down or "
                          "overloaded. Progress is saved; wait 15–30 minutes and run this cell again.")
                    break
                continue
            net_in_a_row = 0
            fetched += 1
            buffer.append(rec)
            failed.pop(match_id, None)
            if not layout_checked:
                health.append(record_is_healthy(rec))
                if len(health) == LAYOUT_SAMPLE:
                    if not layout_ok(health):
                        stop_reason = "layout check failed"
                        raise RuntimeError(layout_msg)
                    layout_checked = True
            # Nothing is saved until the layout check has passed.
            if layout_checked and len(buffer) >= CFG.chunk_size:
                write_chunk(buffer)
                buffer = []
                save_json(FAILED_FILE, failed)
                rate = fetched / max(time.time() - t0, 1)
                left = len(todo) - fetched
                print(f"   saved {fetched}/{len(todo)}  |  {rate * 60:.0f} matches/min  |  "
                      f"≈ {left / max(rate, 1e-9) / 60:.0f} min left overall")
    except BlockedError as e:
        stop_reason = "blocked"
        print(f"⛔ {e}")
    finally:
        if stop_reason != "layout check failed":
            # Fewer than LAYOUT_SAMPLE pages this run: judge the layout on what we have.
            if not layout_checked and health and not layout_ok(health):
                stop_reason = "layout check failed"
                print(f"⛔ {layout_msg}")
            else:
                write_chunk(buffer)
        save_json(FAILED_FILE, failed)
    remaining = len(wanted) - len(load_done() & set(wanted.match_id))
    log.info("Stage B run: fetched=%d stop=%s remaining=%d", fetched, stop_reason, remaining)
    return {"fetched": fetched, "stop_reason": stop_reason, "remaining": remaining,
            "failed_for_good": sum(1 for v in failed.values() if gave_up(v))}

STAGE_B = await run_stage_b()
print(f"\nThis run: {STAGE_B['fetched']} matches saved, stopped because: {STAGE_B['stop_reason']}.")
if STAGE_B["remaining"]:
    print(f"➡️  {STAGE_B['remaining']} matches still to fetch — run this cell again to continue.")
else:
    print("✅ All detail matches downloaded.")

### 1.6 Layer 1 summary
Checks what's on Drive and whether there's enough data to move on. It's fine to continue to Layer 2 before Stage B is 100% complete — later layers use whatever detail is available — but more detail means better map-level features.

In [ ]:
def iter_detail_records(with_html: bool = False):
    for part in sorted(CFG.path("match_details").glob("part-*.jsonl.gz")):
        with gzip.open(part, "rt", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                if not with_html:
                    rec.pop("html", None)
                yield rec

details = {r["match_id"]: r for r in iter_detail_records()}       # later copies win
wanted_ids = set(SERIES[SERIES.tier.isin(CFG.detail_tiers)].match_id)
coverage = len(set(details) & wanted_ids) / max(len(wanted_ids), 1)
healthy = sum(record_is_healthy(r) for r in details.values())
n_maps = sum(len(r["maps"]) for r in details.values())
failed_now = load_json(FAILED_FILE, {})

status = {
    "events": len(EVENTS),
    "series": len(SERIES),
    "series_by_tier": SERIES.tier.value_counts().to_dict(),
    "date_range": [str(SERIES.date.min()), str(SERIES.date.max())],
    "detail_records": len(details),
    "detail_coverage": round(coverage, 3),
    "detail_healthy_share": round(healthy / max(len(details), 1), 3),
    "maps_with_detail": n_maps,
    "failed_matches": len(failed_now),
}
(CFG.path("logs") / "layer1_status.json").write_text(json.dumps(status, indent=2))

def line(ok, text):
    print(("✅ " if ok else "⚠️ ") + text)

line(len(SERIES) >= 500, f"{len(SERIES)} completed series ({status['date_range'][0]} → {status['date_range'][1]})")
line(coverage >= 0.9, f"Detail coverage for {CFG.detail_tiers}: {coverage:.0%} "
                      f"({len(set(details) & wanted_ids)}/{len(wanted_ids)})"
                      + ("" if coverage >= 0.9 else " — re-run 1.5 to continue"))
line(status["detail_healthy_share"] >= 0.95, f"Readable detail pages: {status['detail_healthy_share']:.0%}")
print(f"ℹ️  {n_maps} maps with round scores and player stats; {len(failed_now)} matches failed so far")

if details:
    sample = next(iter(details.values()))
    m0 = sample["maps"][0] if sample["maps"] else {}
    print("\nExample record:")
    print(f"   {sample['teams'][0]['name']} (id {sample['teams'][0]['id']}) vs "
          f"{sample['teams'][1]['name']} (id {sample['teams'][1]['id']}) — {sample['header']['date']}")
    if m0:
        print(f"   Map 1: {m0['map_name']} {m0['score']['team1']}–{m0['score']['team2']}, "
              f"{len(m0['players']['team1'])}+{len(m0['players']['team2'])} players parsed")
    print(f"   Veto: {sample['header']['map_vetos'][:100]}")

LAYER1_OK = len(SERIES) >= 500 and status["detail_healthy_share"] >= 0.95
log.info("Layer 1 summary: %s", status)
print("\n🎉 Layer 1 complete — ready for Layer 2." if LAYER1_OK and coverage >= 0.9 else
      "\n🟡 Layer 1 usable; finish Stage B (re-run 1.5) for full detail." if LAYER1_OK else
      "\nFix the ⚠️ items above before moving on.")

---
# Layer 2 — Cleaning

Turns the raw downloads into **five tidy, trustworthy tables**. Every rule here exists because real data is messy, and a model trained on wrong rows learns wrong lessons.

| Table | One row per | Used for |
|---|---|---|
| `series` | match (Bo1/Bo3/Bo5) | Elo, form, head-to-head, the final prediction |
| `maps` | map played | map-level Elo, map pool strength |
| `player_maps` | player per map | roster strength |
| `vetoes` | ban / pick | predicting which maps get played (Layer 6) |
| `teams` | team | one ID per team, even after renames |

**Layer 2 reads only the files on Drive**, so it never touches vlr.gg. In a new Colab session you can skip Layer 1: run the Layer 0 cells, then run the Layer 2 cells one by one with **Shift + Enter**. (Or just use Run all — Layer 1 skips everything already downloaded, and picks up any new matches.)

### 2.1 Load the raw files
Reads every event's match list (Stage A) and every match detail page (Stage B) from Drive. Takes about a minute — the detail files hold full web pages, which we skip over.

In [ ]:
import re, json, gzip, unicodedata, datetime as dt
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

def read_json(path: Path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return default

START_DATE = dt.date.fromisoformat(CFG.start_date)
EVENTS = read_json(CFG.path("events") / "events.json", {}).get("events", {})
assert EVENTS, "No event list found on Drive — run Layer 1 first."

raw_rows, left_out = [], Counter()
for f in sorted(CFG.path("event_matches").glob("*.json")):
    saved = read_json(f)
    if not saved or saved["event_id"] not in EVENTS:
        continue
    ev = EVENTS[saved["event_id"]]
    tier = classify_event(ev["title"])              # re-checked with today's rules (cell 0.3b)
    if tier is None:
        left_out[ev["title"]] += len(saved["matches"])
        continue
    for m in saved["matches"]:
        raw_rows.append({
            "match_id": str(m.get("match_id", "")), "event_id": saved["event_id"],
            "event": ev["title"], "tier": tier,
            "fetched_at": saved.get("fetched_at", ""),
            "status": m.get("status", ""), "note": m.get("note", ""),
            "date_text": m.get("date", ""), "unix_ts": m.get("unix_timestamp", ""),
            "stage": m.get("stage", ""), "series_name": m.get("event_series", ""),
            "team1": (m["team1"].get("name") or "").strip(), "team2": (m["team2"].get("name") or "").strip(),
            "score1": str(m["team1"].get("score", "")).strip(), "score2": str(m["team2"].get("score", "")).strip(),
            "win1": bool(m["team1"].get("is_winner")), "win2": bool(m["team2"].get("is_winner")),
        })
RAW = pd.DataFrame(raw_rows)

def load_details() -> dict:
    """Latest saved detail record per match, without the raw HTML."""
    out = {}
    for part in sorted(CFG.path("match_details").glob("part-*.jsonl.gz")):
        with gzip.open(part, "rt", encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                rec.pop("html", None)
                prev = out.get(rec["match_id"])
                if prev is None or rec.get("fetched_at", "") >= prev.get("fetched_at", ""):
                    out[rec["match_id"]] = rec
    return out

DETAILS = load_details()
print(f"✅ Loaded {len(RAW):,} raw match rows from {RAW.event_id.nunique()} official events "
      f"and {len(DETAILS):,} detail records.")
kept_events = RAW.groupby("tier").event_id.nunique().to_dict()
print(f"   official events by tier: {kept_events}")
if left_out:
    print(f"   left out {len(left_out)} unofficial events ({sum(left_out.values()):,} match rows), largest:")
    for title, n in left_out.most_common(12):
        print(f"      {n:5,}  {title}")

### 2.2 Clean the series table
Rebuilt from the raw files with **strict rules**. Each rejected row is counted under its reason, so nothing disappears silently.

| Rule | Why |
|---|---|
| Status must be *Completed* | A live match shows a partial score — counting it would record the wrong winner |
| Both teams known, and different | "TBD" rows are unplayed |
| Score must be a series score (1–0, 2–1, 3–2…) **or** a valid map score (13–7) | vlr.gg shows Bo1 matches by their round score, so 13–7 means a Bo1 won 1–0. Ties, 0–0 and impossible scores are removed |
| Winner flag must agree with the score | Catches parsing mistakes |
| Date must contain a year, be ≥ `start_date` and not in the future | Dates come from the page's date text. The parser library's timestamps are **wrong by a year** for older matches, so they're only used when there's no date text |
| Each match ID only once | The same match can appear on two event pages |

In [ ]:
MONTHS = "january|february|march|april|may|june|july|august|september|october|november|december"

def date_from_text(date_text: str):
    """'Wed, September 30, 2026' → date. Only dates that carry a year are trusted."""
    m = re.search(rf"({MONTHS})\s+(\d{{1,2}}),?\s+(20\d\d)", (date_text or "").lower())
    if not m:
        return None
    try:
        return dt.date(int(m.group(3)), MONTHS.split("|").index(m.group(1)) + 1, int(m.group(2)))
    except ValueError:
        return None

def date_from_timestamp(unix_ts: str):
    return dt.date.fromisoformat(unix_ts[:10]) if unix_ts and re.match(r"^\d{4}-\d{2}-\d{2}", unix_ts) else None

def valid_map_score(a: int, b: int) -> bool:
    """13–0…13–11, or an overtime win by exactly 2 (14–12, 15–13…)."""
    hi, lo = max(a, b), min(a, b)
    return (hi == 13 and lo <= 11) or (hi > 13 and hi - lo == 2)

def norm_name(name: str) -> str:
    """Comparison key for team names: no accents, no punctuation, lower case."""
    s = unicodedata.normalize("NFKD", name or "")
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def clean_series(raw: pd.DataFrame):
    reasons, notes, kept, bad_dates = Counter(), Counter(), [], []
    seen = set()
    for r in raw.itertuples(index=False):
        fetched = (dt.date.fromisoformat(r.fetched_at[:10]) if r.fetched_at
                   else dt.datetime.now(dt.timezone.utc).date())
        if not r.match_id:
            reasons["no match id"] += 1; continue
        if r.team1.upper() in ("", "TBD") or r.team2.upper() in ("", "TBD"):
            reasons["teams not decided (TBD)"] += 1; continue
        if r.status.strip().lower() != "completed":
            reasons[f"not completed (status '{r.status or 'blank'}')"] += 1; continue
        if not (r.score1.isdigit() and r.score2.isdigit()):
            reasons["no numeric score"] += 1; continue
        s1, s2 = int(r.score1), int(r.score2)
        if s1 == s2:
            reasons["tied or 0–0 score"] += 1; continue
        rounds1 = rounds2 = None
        if max(s1, s2) not in (1, 2, 3):
            if not valid_map_score(s1, s2):
                reasons["unrecognisable score (neither series nor map score)"] += 1; continue
            rounds1, rounds2 = s1, s2                       # Bo1 shown by its round score
            s1, s2 = (1, 0) if s1 > s2 else (0, 1)
            notes["Bo1 shown as a round score (converted to 1–0)"] += 1
        wins = max(s1, s2)
        winner = 1 if s1 > s2 else 2
        if (r.win1 or r.win2) and (r.win1 != (winner == 1) or r.win2 != (winner == 2)):
            reasons["winner flag disagrees with score"] += 1; continue
        if norm_name(r.team1) == norm_name(r.team2):
            reasons["team plays itself"] += 1; continue
        d_text, d_ts = date_from_text(r.date_text), date_from_timestamp(r.unix_ts)
        d = d_text or d_ts
        if d_text and d_ts and abs((d_text - d_ts).days) > 2:
            notes["timestamp disagreed with the date text (date text used)"] += 1
        if d is None:
            reasons["date has no year / unreadable"] += 1
            bad_dates.append((r.match_id, r.date_text, r.unix_ts, r.status)); continue
        if d < START_DATE:
            reasons["before start date"] += 1; continue
        if d > fetched + dt.timedelta(days=1):
            reasons["date in the future"] += 1
            bad_dates.append((r.match_id, r.date_text, r.unix_ts, r.status)); continue
        if r.match_id in seen:
            reasons["duplicate match id"] += 1; continue
        seen.add(r.match_id)
        kept.append({
            "match_id": r.match_id, "date": d, "event_id": r.event_id, "event": r.event,
            "tier": r.tier, "stage": r.stage, "series_name": r.series_name, "note": r.note,
            "team1_name": r.team1, "team2_name": r.team2, "score1": s1, "score2": s2,
            "winner": winner, "best_of": 2 * wins - 1, "rounds1": rounds1, "rounds2": rounds2,
        })
    df = pd.DataFrame(kept).sort_values(["date", "match_id"]).reset_index(drop=True)
    return df, reasons, notes, bad_dates

SERIES, SERIES_DROPS, SERIES_NOTES, BAD_DATES = clean_series(RAW)

print(f"✅ {len(SERIES):,} clean series kept out of {len(RAW):,} raw rows "
      f"({SERIES.date.min()} → {SERIES.date.max()})")
print("\nRows removed, by reason:")
for reason, n in SERIES_DROPS.most_common():
    print(f"   {n:6,}  {reason}")
if SERIES_NOTES:
    print("\nRows kept but corrected:")
    for note, n in SERIES_NOTES.most_common():
        print(f"   {n:6,}  {note}")
if BAD_DATES:
    print("\nExamples of rows with bad dates (send these to Claude if the count above is large):")
    for mid, text, ts, status in BAD_DATES[:5]:
        print(f"   match {mid}: date text '{text}', timestamp '{ts}', status '{status}'")
print("\nSeries per tier and best-of:")
print(pd.crosstab(SERIES.tier, SERIES.best_of, margins=True))

### 2.3 One ID per team
Stage A (results) only has team **names**; Stage B (details) also has vlr.gg's **team ID**, which stays the same when a team renames itself.

- Every name seen next to an ID becomes an alias of that ID, so *"Leviatán"*, *"Leviatan"* and *"LEVIATÁN"* all map to one team.
- Teams that never appear in a detail page (many Challengers teams) are keyed by their cleaned name.
- If one name was used by two different IDs, the more common one wins and the clash is listed.

In [ ]:
def build_team_identity(series: pd.DataFrame, details: dict):
    name_ids = defaultdict(Counter)          # normalized name -> Counter(vlr ids)
    id_names = defaultdict(Counter)          # vlr id -> Counter(display names)
    id_tags = defaultdict(Counter)
    by_id = series.set_index("match_id")
    for mid, rec in details.items():
        teams = rec.get("teams", [])
        if len(teams) != 2 or mid not in by_id.index:
            continue
        row = by_id.loc[mid]
        stage_a = [row.team1_name, row.team2_name]
        d_norm = [norm_name(t.get("name", "")) for t in teams]
        a_norm = [norm_name(n) for n in stage_a]
        if d_norm == a_norm[::-1] and d_norm != a_norm:
            stage_a = stage_a[::-1]                       # detail page lists teams the other way round
        for t, a_name in zip(teams, stage_a):
            tid = str(t.get("id") or "")
            if not tid:
                continue
            for nm in {t.get("name", ""), a_name}:
                if nm:
                    name_ids[norm_name(nm)][tid] += 1
                    id_names[tid][nm] += 1
            if t.get("tag"):
                id_tags[tid][t["tag"]] += 1

    name_to_id, clashes = {}, []
    for nm, ids in name_ids.items():
        (best, n_best), *rest = ids.most_common()
        name_to_id[nm] = best
        if rest:
            clashes.append((nm, dict(ids)))
    return name_to_id, id_names, id_tags, clashes

NAME_TO_ID, ID_NAMES, ID_TAGS, ID_CLASHES = build_team_identity(SERIES, DETAILS)

def team_key(name: str) -> str:
    n = norm_name(name)
    return f"vlr:{NAME_TO_ID[n]}" if n in NAME_TO_ID else f"name:{n}"

SERIES["team1"] = SERIES.team1_name.map(team_key)
SERIES["team2"] = SERIES.team2_name.map(team_key)
SERIES["winner_team"] = np.where(SERIES.winner == 1, SERIES.team1, SERIES.team2)

# Team table: latest display name, all aliases, tags, activity.
sides = pd.concat([
    SERIES[["team1", "team1_name", "date", "tier"]].set_axis(["key", "name", "date", "tier"], axis=1),
    SERIES[["team2", "team2_name", "date", "tier"]].set_axis(["key", "name", "date", "tier"], axis=1)])
sides = sides.sort_values("date")
TEAMS = sides.groupby("key").agg(
    name=("name", "last"), first_seen=("date", "min"), last_seen=("date", "max"),
    n_series=("date", "size"), tiers=("tier", lambda s: "|".join(sorted(set(s)))),
    aliases=("name", lambda s: "|".join(sorted(set(s))))).reset_index()
TEAMS["vlr_id"] = TEAMS.key.str.extract(r"^vlr:(\d+)$")[0]
TEAMS["tag"] = TEAMS.vlr_id.map(lambda i: ID_TAGS[i].most_common(1)[0][0] if i in ID_TAGS else "")

with_id = (SERIES.team1.str.startswith("vlr:").mean() + SERIES.team2.str.startswith("vlr:").mean()) / 2
print(f"✅ {len(TEAMS):,} distinct teams; {TEAMS.vlr_id.notna().sum():,} have a vlr.gg ID.")
print(f"   Team sides with an ID: {with_id:.0%} overall")
for tier, g in SERIES.groupby("tier"):
    share = (g.team1.str.startswith("vlr:").mean() + g.team2.str.startswith("vlr:").mean()) / 2
    print(f"      {tier:14s} {share:.0%}")
renamed = TEAMS[TEAMS.aliases.str.contains(r"\|")]
print(f"   {len(renamed)} teams appear under more than one name, e.g.:")
for r in renamed.sort_values("n_series", ascending=False).head(5).itertuples():
    print(f"      {r.name:25s} ← {r.aliases.replace('|', ', ')[:80]}")
if ID_CLASHES:
    print(f"   ⚠️ {len(ID_CLASHES)} names were used by more than one team ID (most common kept): "
          f"{[c[0] for c in ID_CLASHES[:5]]}")

### 2.4 Maps, players and vetoes
For every match with a detail page:

- **Line the teams up** with the series row (detail pages sometimes list them the other way round).
- **Check every map score is a real Valorant score:** 13–0…13–11, or an overtime win by exactly 2 (14–12, 15–13…).
- **Check the map wins add up to the series score.**
- If either check fails, **all of that match's maps are left out** (one bad score means the page can't be trusted) — the series result itself stays.
- **No maps on the page** → the match was a **forfeit**. It is flagged, and later layers leave it out of skill ratings.
- **Vetoes:** `"NS ban Summit; NS pick Sunset; … Haven remains"` → one row per action, linked to a team. This also tells us who **picked** each map. The veto's short tags don't always match the tag shown on the match page, so each tag is resolved in this order:
  1. it equals a team's tag on the page;
  2. by elimination — if the other tag is known, this one is the opponent;
  3. **learned** — across all matches, a team's own tag keeps appearing with that team, while its opponents change.
- **Patch** (`"Patch 13.04"`) is read from the match header.

In [ ]:
def to_int(x):
    return int(x) if isinstance(x, int) or (isinstance(x, str) and x.strip().isdigit()) else None

def to_num(x):
    """'1.20' → 1.2, '75%' → 0.75, '+5' → 5.0, '' → NaN."""
    if x is None:
        return np.nan
    s = str(x).strip().replace("−", "-")
    pct = s.endswith("%")
    s = s.rstrip("%").lstrip("+")
    try:
        v = float(s)
    except ValueError:
        return np.nan
    return v / 100 if pct else v

def header_month_day(text: str):
    m = re.search(rf"({MONTHS})\s+(\d{{1,2}})\b", (text or "").lower())
    return (MONTHS.split("|").index(m.group(1)) + 1, int(m.group(2))) if m else None

def parse_veto(text: str):
    steps = []
    for i, tok in enumerate(t.strip() for t in (text or "").split(";") if t.strip()):
        m = re.match(r"^(.+?)\s+(ban|pick)\s+(.+)$", tok, flags=re.I)
        if m:
            steps.append((i + 1, m.group(1).strip(), m.group(2).lower(), m.group(3).strip()))
            continue
        m = re.match(r"^(.+?)\s+remains$", tok, flags=re.I)
        if m:
            steps.append((i + 1, None, "remains", m.group(1).strip()))
    return steps

PLAYER_STATS = ["rating", "acs", "kills", "deaths", "assists", "kd_diff", "kast", "adr",
                "hs_pct", "fk", "fd", "fk_diff"]

def learn_veto_tags(srow: pd.DataFrame, details: dict) -> dict:
    """tag → Counter(team key): how often each veto tag appears in a match involving each team."""
    counts = defaultdict(Counter)
    for mid, rec in details.items():
        if mid not in srow.index:
            continue
        row = srow.loc[mid]
        for tag in {t.lower() for _, t, _, _ in parse_veto(rec.get("header", {}).get("map_vetos", "")) if t}:
            counts[tag][row.team1] += 1
            counts[tag][row.team2] += 1
    return counts

def resolve_veto_tags(vtags: list, keys: list, page_side: dict, counts: dict):
    """Map each veto tag to series side 0/1. Returns (tag → side, method)."""
    side_of = {t: page_side[t] for t in vtags if t in page_side}
    unknown = [t for t in vtags if t not in side_of]
    method = "page tag" if side_of else ""
    if not unknown:
        return side_of, method or "none"
    if len(unknown) == 1 and len(set(side_of.values())) == 1:
        side_of[unknown[0]] = 1 - next(iter(side_of.values()))
        return side_of, "elimination"
    if len(unknown) == 2 and not side_of:
        a, b = unknown
        straight = counts[a][keys[0]] + counts[b][keys[1]]
        swapped = counts[a][keys[1]] + counts[b][keys[0]]
        if straight != swapped:
            side_of.update({a: 0, b: 1} if straight > swapped else {a: 1, b: 0})
            return side_of, "learned"
        return side_of, "unresolved"
    for t in unknown:                                      # odd cases: judge each tag on its own
        c0, c1 = counts[t][keys[0]], counts[t][keys[1]]
        if c0 != c1 and (0 if c0 > c1 else 1) not in side_of.values():
            side_of[t] = 0 if c0 > c1 else 1
    return side_of, "learned" if len(side_of) == len(vtags) else "unresolved"

def build_detail_tables(series: pd.DataFrame, details: dict):
    srow = series.set_index("match_id")
    maps, players, vetoes, issues = [], [], [], Counter()
    match_info = {}
    tag_counts = learn_veto_tags(srow, details)
    veto_methods, unresolved_examples = Counter(), []
    for mid, rec in details.items():
        if mid not in srow.index:
            issues["detail page for a series that was removed in 2.2"] += 1; continue
        row = srow.loc[mid]
        teams = rec.get("teams", [])
        if len(teams) != 2:
            issues["detail page without two teams"] += 1; continue
        d_keys = [f"vlr:{t['id']}" if t.get("id") else team_key(t.get("name", "")) for t in teams]
        if d_keys == [row.team1, row.team2]:
            flip = False
        elif d_keys == [row.team2, row.team1]:
            flip = True
        else:
            issues["teams on detail page don't match the series"] += 1; continue
        keys = [row.team1, row.team2]
        header = rec.get("header", {})
        patch_m = re.search(r"patch\s*(\d+\.\d+)", header.get("date", ""), flags=re.I)
        md_ = header_month_day(header.get("date", ""))
        date_ok = None
        if md_:
            cands = []
            for y in (row.date.year - 1, row.date.year, row.date.year + 1):
                try:
                    cands.append(dt.date(y, *md_))
                except ValueError:                                # e.g. 29 February in a non-leap year
                    pass
            date_ok = bool(cands) and min(abs((c - row.date).days) for c in cands) <= 1

        # tags → series side, for the veto
        tag_side = {}
        for side, t in enumerate(teams):
            if t.get("tag"):
                tag_side[t["tag"].strip().lower()] = (1 - side) if flip else side

        played, map_wins = [], [0, 0]
        for k, m in enumerate(rec.get("maps", [])):
            a, b = to_int(m["score"].get("team1")), to_int(m["score"].get("team2"))
            if a is None or b is None:
                continue                                          # map listed but not played
            if flip:
                a, b = b, a
            ok = valid_map_score(a, b)
            sides = {}
            for half in ("score_ct", "score_t", "score_ot"):
                h = m.get(half, {})
                h1, h2 = to_num(h.get("team1")), to_num(h.get("team2"))
                sides[half] = (h2, h1) if flip else (h1, h2)
            played.append({"match_id": mid, "map_number": len(played) + 1,
                           "map_name": (m.get("map_name") or "").strip(),
                           "team1": keys[0], "team2": keys[1], "score1": a, "score2": b,
                           "winner_team": keys[0] if a > b else keys[1],
                           "t1_ct": sides["score_ct"][0], "t1_t": sides["score_t"][0],
                           "t2_ct": sides["score_ct"][1], "t2_t": sides["score_t"][1],
                           "overtime": max(a, b) > 13, "valid": ok, "_players": m.get("players", {})})
            map_wins[0 if a > b else 1] += 1

        forfeit = len(played) == 0
        all_valid = all(mp["valid"] for mp in played)
        adds_up = map_wins == [row.score1, row.score2]
        consistent = (not forfeit) and all_valid and adds_up
        if not forfeit and not all_valid:
            issues["impossible map score, e.g. 13–12 (match's maps dropped)"] += 1
        elif not forfeit and not adds_up:
            issues["map wins don't add up to series score (match's maps dropped)"] += 1

        # veto → who picked each map
        steps = parse_veto(header.get("map_vetos", ""))
        vtags = list(dict.fromkeys(t.lower() for _, t, _, _ in steps if t))
        side_of, method = resolve_veto_tags(vtags, keys, tag_side, tag_counts)
        if vtags:
            veto_methods[method] += 1
            if method == "unresolved" and len(unresolved_examples) < 3:
                unresolved_examples.append((header.get("map_vetos", "")[:70],
                                            [t.get("tag", "") for t in teams]))
        picker = {}
        for order, tag, action, map_name in steps:
            side = side_of.get(tag.lower()) if tag else None
            team = keys[side] if side is not None else None
            if tag and side is None:
                issues["veto tag not matched to a team"] += 1
            vetoes.append({"match_id": mid, "order": order, "team": team, "action": action,
                           "map_name": map_name})
            if action == "pick" and team:
                picker[map_name.lower()] = team
            elif action == "remains":
                picker[map_name.lower()] = "decider"

        if consistent:
            for mp in played:
                mp["picked_by"] = picker.get(mp["map_name"].lower(), "")
                pl = mp.pop("_players")
                sides_pl = [pl.get("team1", []), pl.get("team2", [])]
                if flip:
                    sides_pl = sides_pl[::-1]
                mp["players_complete"] = all(len(s) == 5 for s in sides_pl)
                for side, plist in enumerate(sides_pl):
                    for p in plist:
                        players.append({"match_id": mid, "map_number": mp["map_number"],
                                        "team": keys[side], "player": (p.get("name") or "").strip(),
                                        "agent": p.get("agent", ""),
                                        **{s: to_num(p.get(s)) for s in PLAYER_STATS}})
                maps.append(mp)
        match_info[mid] = {"has_detail": True, "forfeit": forfeit, "maps_ok": consistent,
                           "patch": patch_m.group(1) if patch_m else "",
                           "detail_date_ok": date_ok, "veto_steps": len(steps)}
    map_cols = ["match_id", "map_number", "map_name", "team1", "team2", "score1", "score2",
                "winner_team", "t1_ct", "t1_t", "t2_ct", "t2_t", "overtime", "picked_by",
                "players_complete"]
    maps_df = pd.DataFrame(maps, columns=map_cols + ["_players", "valid"]).drop(columns=["_players", "valid"])
    players_df = pd.DataFrame(players, columns=["match_id", "map_number", "team", "player", "agent"] + PLAYER_STATS)
    vetoes_df = pd.DataFrame(vetoes, columns=["match_id", "order", "team", "action", "map_name"])
    issues.update({f"veto tags resolved by: {k}": v for k, v in veto_methods.items()})
    return maps_df, players_df, vetoes_df, match_info, issues, unresolved_examples

MAPS, PLAYER_MAPS, VETOES, MATCH_INFO, DETAIL_ISSUES, VETO_UNRESOLVED = build_detail_tables(SERIES, DETAILS)

info = pd.DataFrame.from_dict(MATCH_INFO, orient="index").reindex(
    columns=["has_detail", "forfeit", "maps_ok", "patch", "detail_date_ok", "veto_steps"])
SERIES = SERIES.merge(info, left_on="match_id", right_index=True, how="left")
SERIES["has_detail"] = SERIES.has_detail.fillna(False).astype(bool)
SERIES["forfeit"] = SERIES.forfeit.fillna(False).astype(bool)
SERIES["maps_ok"] = SERIES.maps_ok.fillna(False).astype(bool)
SERIES["patch"] = SERIES.patch.fillna("")
SERIES["veto_steps"] = SERIES.veto_steps.fillna(0).astype(int)
MAPS = MAPS.merge(SERIES[["match_id", "date", "tier", "event_id", "patch"]], on="match_id", how="left")

d = SERIES[SERIES.has_detail]
print(f"✅ {len(MAPS):,} maps, {len(PLAYER_MAPS):,} player-map rows, {len(VETOES):,} veto steps "
      f"from {len(d):,} detailed matches")
print(f"   forfeits: {int(d.forfeit.sum())}  |  matches with usable maps: {d.maps_ok.mean():.1%}  |  "
      f"patch found: {(d.patch != '').mean():.0%}  |  veto found: {(d.veto_steps > 0).mean():.0%}")
if DETAIL_ISSUES:
    print("   Issues found (and handled):")
    for issue, n in DETAIL_ISSUES.most_common():
        print(f"      {n:5,}  {issue}")
for veto_text, page_tags in VETO_UNRESOLVED:
    print(f"   unresolved veto example: '{veto_text}…' — tags on the match page: {page_tags}")
print("\nMaps played most often:")
print(MAPS.map_name.value_counts().head(12).to_string())

### 2.5 Cross-checks and save
Final consistency checks, then all tables are saved to `data/clean/` on Drive, plus a **quality report** (`logs/layer2_quality.json`) recording every number above.

In [ ]:
checks = []
def check(ok: bool, text: str, warn_only: bool = False):
    checks.append((ok, text, warn_only))
    print(("✅ " if ok else ("⚠️ " if warn_only else "❌ ")) + text)

d = SERIES[SERIES.has_detail & ~SERIES.forfeit]
date_known = d.detail_date_ok.dropna()
check(SERIES.match_id.is_unique, "every series appears exactly once")
check((SERIES.date <= dt.date.today() + dt.timedelta(days=1)).all(), "no series dated in the future")
check((SERIES.winner_team == np.where(SERIES.score1 > SERIES.score2, SERIES.team1, SERIES.team2)).all(),
      "winner matches the score for every series")
check(len(d) == 0 or d.maps_ok.mean() >= 0.95,
      f"{d.maps_ok.mean():.1%} of detailed matches have maps that add up to the series score (need ≥ 95%)")
check(len(date_known) == 0 or date_known.mean() >= 0.95,
      f"{date_known.mean():.1%} of detail pages agree with the results date (±1 day; need ≥ 95%)")
if len(MAPS):
    check(MAPS.apply(lambda r: valid_map_score(r.score1, r.score2), axis=1).all(), "every kept map score is valid")
    halves = MAPS[["t1_ct", "t1_t", "t2_ct", "t2_t"]].notna().all(axis=1).mean()
    check(halves >= 0.9, f"{halves:.0%} of maps have attack/defence half scores", warn_only=True)
    full = MAPS.players_complete.mean()
    check(full >= 0.95, f"{full:.0%} of maps have all 10 players' stats", warn_only=True)
    picked = (MAPS.picked_by != "").mean()
    check(picked >= 0.8, f"{picked:.0%} of maps know who picked them (or that it was the decider)", warn_only=True)
id_share = (SERIES.team1.str.startswith("vlr:") | SERIES.team2.str.startswith("vlr:")).mean()
check(id_share >= 0.3, f"{id_share:.0%} of series involve at least one team with a vlr.gg ID", warn_only=True)

out = CFG.path("clean")
for name, df in [("series", SERIES), ("maps", MAPS), ("player_maps", PLAYER_MAPS),
                 ("vetoes", VETOES), ("teams", TEAMS)]:
    df.to_parquet(out / f"{name}.parquet", index=False)

quality = {
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "raw_rows": len(RAW), "series": len(SERIES), "series_drops": dict(SERIES_DROPS),
    "series_corrections": dict(SERIES_NOTES),
    "bad_date_examples": BAD_DATES[:20], "teams": len(TEAMS),
    "teams_with_vlr_id": int(TEAMS.vlr_id.notna().sum()), "id_clashes": ID_CLASHES[:50],
    "detail_issues": dict(DETAIL_ISSUES), "maps": len(MAPS), "player_maps": len(PLAYER_MAPS),
    "vetoes": len(VETOES), "checks": [{"ok": bool(o), "check": t} for o, t, _ in checks],
}
(CFG.path("logs") / "layer2_quality.json").write_text(json.dumps(quality, indent=2, default=str))

LAYER2_OK = all(ok or warn for ok, _, warn in checks)
log.info("Layer 2 done: series=%d maps=%d players=%d ok=%s",
         len(SERIES), len(MAPS), len(PLAYER_MAPS), LAYER2_OK)
print(f"\nSaved to {out}: series, maps, player_maps, vetoes, teams")
print("\n🎉 Layer 2 complete — ready for Layer 3." if LAYER2_OK else
      "\nSome ❌ checks failed — send this output to Claude before moving on.")

---
# Layer 3 — Features

For every match, this layer writes down what was known about both teams **before the match was played**. Those numbers (*features*) are what the model learns from.

**The one rule that matters most: no peeking at the future.** If a feature accidentally includes the match's own result, the model looks brilliant in testing and is useless in real life. So the engine walks through history **one day at a time**:

1. Take a snapshot of every team's numbers as they stood the night before.
2. Record the snapshot for each of today's matches.
3. Only then update the numbers with today's results.

Two matches on the same day can't see each other — even if one was played earlier, because we don't trust match times enough to order them. **Section 3.4 then proves it** with an automatic test.

| Feature | What it measures |
|---|---|
| **Elo** | Overall strength. Beat a strong team → big gain; lose to a weak one → big drop. Close wins count a bit less. |
| **Region strength** | Teams in a league mostly play each other, so a 1650 in Pacific and a 1650 in EMEA aren't automatically equal. Each region (Americas, EMEA, Pacific, China) gets an offset learned only from **international** matches; *adjusted Elo* = Elo + region offset. |
| **Map Elo + per-map strength** | Strength on maps in general, and on each specific map |
| **Form** | Recent win rate, with older results fading out (half-life 60 days) |
| **Round difference** | How convincingly a team wins or loses its maps lately |
| **Roster rating** | Average recent rating of the 5 players in the team's latest lineup |
| **Roster continuity** | Share of that lineup who also played in the team's recent matches — low after roster changes |
| **Head-to-head** | Past results between these two teams, smoothed so a single 1–0 isn't "100%" |
| **Tier experience** | Share of a team's matches played in VCT (not Challengers) |
| **Patch** | Days since the current game patch, and each team's record on it |
| **Context** | Tier, Bo1/Bo3/Bo5, playoffs, days of rest |

### 3.1 Load the clean tables and the feature settings
All tunable numbers live in `FeatureParams`. Layer 5 can re-run the engine with different values (using only training data) to find better ones — the defaults below are sensible starting points.

**Why different starting Elo by tier?** VCT and Challengers teams rarely play each other, so their ratings would otherwise drift on separate scales: an unbeaten Challengers team could end up "rated" above a VCT team it would lose to. Starting Challengers teams lower (1400 vs 1600) keeps the two pools roughly comparable; the *tier experience* feature lets the model correct the rest.

In [ ]:
import math, time, json, datetime as dt
from collections import defaultdict, Counter
from dataclasses import dataclass, asdict, field
import numpy as np
import pandas as pd

clean = CFG.path("clean")
SERIES_C = pd.read_parquet(clean / "series.parquet")
MAPS_C = pd.read_parquet(clean / "maps.parquet")
PLAYERS_C = pd.read_parquet(clean / "player_maps.parquet")
TEAMS_C = pd.read_parquet(clean / "teams.parquet")
SERIES_C["date"] = pd.to_datetime(SERIES_C["date"])
MAPS_C["date"] = pd.to_datetime(MAPS_C["date"])

# A player listed twice for the same team on the same map (a page quirk) would make results depend on
# row order. Merge such duplicates into one row with their average stats.
_pkey = ["match_id", "map_number", "team", "player"]
N_PLAYER_DUPES = int(PLAYERS_C.duplicated(_pkey).sum())
if N_PLAYER_DUPES:
    _num = [c for c in PLAYERS_C.columns if c not in _pkey and pd.api.types.is_numeric_dtype(PLAYERS_C[c])]
    _agg = {c: "mean" for c in _num} | {c: "first" for c in PLAYERS_C.columns if c not in _pkey and c not in _num}
    PLAYERS_C = PLAYERS_C.groupby(_pkey, as_index=False, sort=False).agg(_agg)

@dataclass
class FeatureParams:
    k_series: float = 32.0            # Elo step size per series
    close_win_factor: float = 0.75    # a 2–1 / 3–2 / close Bo1 moves Elo less than a 2–0
    k_map: float = 24.0               # map Elo step size
    k_map_specific: float = 12.0      # step size for the per-map adjustment
    init_tier1: float = 1600.0        # starting Elo for teams first seen in VCT
    init_challengers: float = 1400.0  # starting Elo for teams first seen in Challengers
    season_regress: float = 0.25      # each new year, pull ratings 25% back toward 1500 (rosters change)
    k_region: float = 8.0             # region-offset step size per international series between regions
    form_halflife_days: float = 60.0
    form_prior_games: float = 2.0     # form starts at 50% "worth" 2 games
    h2h_halflife_days: float = 365.0
    player_halflife_maps: float = 20.0
    player_prior_maps: float = 5.0    # new players start near an average rating (1.0)
    rest_cap_days: int = 60

PARAMS = FeatureParams()
print(f"✅ Loaded {len(SERIES_C):,} series, {len(MAPS_C):,} maps, {len(PLAYERS_C):,} player-map rows")
if N_PLAYER_DUPES:
    print(f"   merged {N_PLAYER_DUPES:,} duplicate player rows (same player, team and map listed twice)")
print("   Feature settings:", asdict(PARAMS))

### 3.2 The feature engine
One object holds every team's current state. `snapshot()` reads it (never changes it); `update_*()` changes it. The build loop in 3.3 always calls snapshot for a whole day before any update for that day.

In [ ]:
TIER1 = {"international", "regional"}

def _decay(halflife, days):
    return 0.5 ** (days / halflife) if halflife > 0 else 1.0

class FeatureEngine:
    def __init__(self, p: FeatureParams):
        self.p = p
        self.elo, self.map_base = {}, {}
        self.map_adj = defaultdict(float)            # (team, map) → per-map adjustment
        self.map_games = Counter()                   # (team, map) → maps played
        self.n_series, self.n_tier1 = Counter(), Counter()
        self.last_date = {}
        self.form = {}                               # team → [weighted wins, weight, as-of date]
        self.round_diff = {}                         # team → [weighted round diff, weight, as-of date]
        self.h2h = defaultdict(list)                 # (teamA, teamB) sorted → [(date, winner)]
        self.player = {}                             # player → [weighted rating sum, weight]
        self.lineups = defaultdict(list)             # team → [(date, frozenset of players)]
        self.patch_first, self.current_patch = {}, None
        self.patch_record = {}                       # (team, patch) → [wins, games] (plain dict so it can be saved)
        self.team_region = {}                        # team → "americas" / "emea" / "pacific" / "china"
        self.region_offset = {}                      # region → rating offset learned from international play
        self.year = None
        self._day_patches = []

    # ---------- reading state (never modifies it) ----------
    def _start_rating(self, tier):
        return self.p.init_tier1 if tier in TIER1 else self.p.init_challengers

    def _ew(self, store, team, date, prior_w=0.0, prior_v=0.0, halflife=None):
        if team not in store:
            return (prior_v if prior_w > 0 else np.nan), 0.0
        s, w, d0 = store[team]
        f = _decay(halflife or self.p.form_halflife_days, (date - d0).days)
        s, w = s * f, w * f
        return ((s + prior_w * prior_v) / (w + prior_w) if (w + prior_w) > 0 else np.nan), w

    def offset(self, team):
        region = self.team_region.get(team)
        return self.region_offset.get(region, 0.0) if region else 0.0

    def player_rating(self, name):
        s, w = self.player.get(name, (0.0, 0.0))
        return (s + self.p.player_prior_maps * 1.0) / (w + self.p.player_prior_maps)

    def team_snapshot(self, team, tier, date) -> dict:
        p = self.p
        elo = self.elo.get(team, self._start_rating(tier))
        n = self.n_series[team]
        form, _ = self._ew(self.form, team, date, prior_w=p.form_prior_games, prior_v=0.5)
        rdiff, rw = self._ew(self.round_diff, team, date)
        last = self.last_date.get(team)
        off = self.offset(team)
        f = {
            "elo": elo,
            "region_offset": off,
            "elo_adj": elo + off,
            "map_elo": self.map_base.get(team, elo),
            "n_series": n,
            "days_rest": min((date - last).days, p.rest_cap_days) if last is not None else p.rest_cap_days,
            "is_new": int(n == 0),
            "form": form,
            "round_diff": rdiff if rw > 0 else np.nan,
            "tier1_share": self.n_tier1[team] / n if n else np.nan,
        }
        lineups = self.lineups.get(team)
        if lineups:
            lu_date, last_lu = lineups[-1]
            # sorted: a set has no fixed order, and adding decimals in a different order can change the last digit
            f["roster_rating"] = float(np.mean([self.player_rating(x) for x in sorted(last_lu)]))
            f["roster_days_old"] = (date - lu_date).days
            prev = lineups[-11:-1]
            if len(prev) >= 3:
                seen = set().union(*(lu for _, lu in prev))
                f["roster_continuity"] = len(last_lu & seen) / len(last_lu)
            else:
                f["roster_continuity"] = np.nan
        else:
            f["roster_rating"] = f["roster_days_old"] = f["roster_continuity"] = np.nan
        if self.current_patch:
            w, g = self.patch_record.get((team, self.current_patch), (0, 0))
            f["patch_winrate"] = (w + 1) / (g + 2)
            f["patch_games"] = g
        else:
            f["patch_winrate"], f["patch_games"] = np.nan, 0
        return f

    def pair_snapshot(self, t1, t2, date) -> dict:
        hist = self.h2h.get(tuple(sorted((t1, t2))), [])
        w1 = tot = 0.0
        for d, winner in hist:
            f = _decay(self.p.h2h_halflife_days, (date - d).days)
            tot += f
            w1 += f * (winner == t1)
        return {"h2h_n": len(hist), "h2h_t1": (w1 + 1) / (tot + 2)}

    def map_snapshot(self, team, tier, map_name) -> dict:
        base = self.map_base.get(team, self.elo.get(team, self._start_rating(tier)))
        return {"map_rating": base + self.map_adj.get((team, map_name), 0.0) + self.offset(team),
                "map_games": self.map_games[(team, map_name)]}

    def days_since_patch(self, date):
        return (date - self.patch_first[self.current_patch]).days if self.current_patch else np.nan

    # ---------- changing state ----------
    def start_day(self, date):
        if self.year is not None and date.year != self.year:        # new season
            r = self.p.season_regress
            for store in (self.elo, self.map_base):
                for t in store:
                    store[t] = 1500 + (store[t] - 1500) * (1 - r)
            for k in self.map_adj:
                self.map_adj[k] *= 0.5
            for k in self.region_offset:
                self.region_offset[k] *= (1 - r)
        self.year = date.year
        self._day_patches = []

    def _ensure(self, team, tier):
        if team not in self.elo:
            self.elo[team] = self.map_base[team] = self._start_rating(tier)

    def _ew_add(self, store, team, date, value, halflife=None):
        if team in store:
            s, w, d0 = store[team]
            f = _decay(halflife or self.p.form_halflife_days, (date - d0).days)
            store[team] = [s * f + value, w * f + 1.0, date]
        else:
            store[team] = [value, 1.0, date]

    def update_series(self, r):
        """r: one clean series row. Forfeits are never passed in."""
        p, t1, t2 = self.p, r.team1, r.team2
        self._ensure(t1, r.tier); self._ensure(t2, r.tier)
        if r.tier == "regional" and event_region(r.event):        # a league match tells us the teams' region
            self.team_region[t1] = self.team_region[t2] = event_region(r.event)
        e1, e2 = self.elo[t1] + self.offset(t1), self.elo[t2] + self.offset(t2)
        expected = 1 / (1 + 10 ** ((e2 - e1) / 400))
        won = 1.0 if r.winner == 1 else 0.0
        if r.best_of > 1:
            close = min(r.score1, r.score2) == max(r.score1, r.score2) - 1
        else:
            rd = abs((r.rounds1 or 0) - (r.rounds2 or 0)) if pd.notna(r.rounds1) and pd.notna(r.rounds2) else 99
            close = rd <= 3
        k = p.k_series * (p.close_win_factor if close else 1.0)
        self.elo[t1] += k * (won - expected)
        self.elo[t2] -= k * (won - expected)
        r1, r2 = self.team_region.get(t1), self.team_region.get(t2)
        if r.tier == "international" and r1 and r2 and r1 != r2:   # only cross-region play moves offsets
            self.region_offset[r1] = self.region_offset.get(r1, 0.0) + p.k_region * (won - expected)
            self.region_offset[r2] = self.region_offset.get(r2, 0.0) - p.k_region * (won - expected)
        for team, w in ((t1, won), (t2, 1 - won)):
            self.n_series[team] += 1
            self.n_tier1[team] += r.tier in TIER1
            self.last_date[team] = r.date
            self._ew_add(self.form, team, r.date, w)
            if self.current_patch:
                rec = self.patch_record.setdefault((team, self.current_patch), [0, 0])
                rec[0] += w; rec[1] += 1
        self.h2h[tuple(sorted((t1, t2)))].append((r.date, r.winner_team))
        if getattr(r, "patch", ""):
            self._day_patches.append(r.patch)

    def update_map(self, m):
        p, t1, t2, name = self.p, m.team1, m.team2, m.map_name
        for t in (t1, t2):
            self.map_base.setdefault(t, self.elo.get(t, 1500.0))
        r1 = self.map_base[t1] + self.map_adj[(t1, name)] + self.offset(t1)
        r2 = self.map_base[t2] + self.map_adj[(t2, name)] + self.offset(t2)
        expected = 1 / (1 + 10 ** ((r2 - r1) / 400))
        won = 1.0 if m.score1 > m.score2 else 0.0
        margin = 0.75 + 0.5 * min(abs(m.score1 - m.score2), 13) / 13
        delta = won - expected
        self.map_base[t1] += p.k_map * margin * delta
        self.map_base[t2] -= p.k_map * margin * delta
        self.map_adj[(t1, name)] += p.k_map_specific * margin * delta
        self.map_adj[(t2, name)] -= p.k_map_specific * margin * delta
        self.map_games[(t1, name)] += 1
        self.map_games[(t2, name)] += 1
        self._ew_add(self.round_diff, t1, m.date, m.score1 - m.score2)
        self._ew_add(self.round_diff, t2, m.date, m.score2 - m.score1)

    def update_players(self, team, date, map_rows):
        """map_rows: this team's player rows for one match, in map order."""
        dec = _decay(self.p.player_halflife_maps, 1)
        lineup = None
        for _, g in map_rows.groupby("map_number", sort=True):
            g = g.sort_values(["player", "rating"], na_position="last")      # fixed order → same result every time
            names = [n.strip().lower() for n in g.player if isinstance(n, str) and n.strip()]
            if lineup is None and names:
                lineup = frozenset(names)
            for name, rating in zip(g.player, g.rating):
                if not isinstance(name, str) or pd.isna(rating):
                    continue
                key = name.strip().lower()
                s, w = self.player.get(key, (0.0, 0.0))
                self.player[key] = (s * dec + float(rating), w * dec + 1.0)
        if lineup:
            self.lineups[team].append((date, lineup))

    # ---------- saving / loading as plain data (works across notebook versions) ----------
    def to_state(self) -> dict:
        state = {k: (dict(v) if isinstance(v, (defaultdict, Counter)) else v)
                 for k, v in vars(self).items() if k != "p"}
        return {"params": asdict(self.p), "state": state}

    @classmethod
    def from_state(cls, saved: dict) -> "FeatureEngine":
        eng = cls(FeatureParams(**saved["params"]))
        for k, v in saved["state"].items():
            cur = getattr(eng, k, None)
            if isinstance(cur, Counter):
                v = Counter(v)
            elif isinstance(cur, defaultdict):
                v = defaultdict(cur.default_factory, v)
            setattr(eng, k, v)
        return eng

    def end_day(self, date):
        for patch in self._day_patches:
            self.patch_first.setdefault(patch, date)
        if self.patch_first:
            newest = max(self.patch_first, key=lambda x: (self.patch_first[x],
                                                          tuple(int(v) for v in x.split(".") if v.isdigit())))
            self.current_patch = newest

print("✅ Feature engine defined.")

### 3.3 Build the features
Walks through all 3+ years of matches day by day (≈ 10–30 seconds). Produces:
- **`series_features.parquet`** — one row per series: both teams' snapshot (`t1_…`, `t2_…`), head-to-head, context, and the result.
- **`map_features.parquet`** — one row per played map: both teams' strength on that map, who picked it, and the result.
- **`engine_state.joblib`** — the engine's final state, so Layer 9 can compute features for *upcoming* matches without replaying history.

Forfeits get a feature row (marked `forfeit = 1`) but never change anyone's ratings, and later layers leave them out of training.

In [ ]:
import joblib

TEAM_FEATURES = ["elo", "region_offset", "elo_adj", "map_elo", "n_series", "days_rest", "is_new", "form", "round_diff", "tier1_share",
                 "roster_rating", "roster_days_old", "roster_continuity", "patch_winrate", "patch_games"]

def build_features(series: pd.DataFrame, maps: pd.DataFrame, players: pd.DataFrame, p: FeatureParams):
    eng = FeatureEngine(p)
    series = series.sort_values(["date", "match_id"])
    maps_by = {k: g.sort_values("map_number") for k, g in maps.groupby("match_id")}
    players_by = {k: g for k, g in players.groupby(["match_id", "team"])}
    s_rows, m_rows = [], []
    for date, day in series.groupby("date", sort=True):
        eng.start_day(date)
        # 1) snapshots for the whole day — nothing from today is known yet
        for r in day.itertuples(index=False):
            a = eng.team_snapshot(r.team1, r.tier, date)
            b = eng.team_snapshot(r.team2, r.tier, date)
            row = {"match_id": r.match_id, "date": date, "tier": r.tier, "best_of": r.best_of,
                   "event_id": r.event_id, "team1": r.team1, "team2": r.team2,
                   "is_playoffs": int("playoff" in str(r.stage).lower() or "final" in str(r.series_name).lower()),
                   "days_since_patch": eng.days_since_patch(date),
                   "forfeit": int(bool(r.forfeit)), "has_detail": int(bool(r.has_detail)),
                   "t1_win": int(r.winner == 1)}
            row.update({f"t1_{k}": v for k, v in a.items()})
            row.update({f"t2_{k}": v for k, v in b.items()})
            row.update(eng.pair_snapshot(r.team1, r.team2, date))
            row["elo_p_t1"] = 1 / (1 + 10 ** ((b["elo_adj"] - a["elo_adj"]) / 400))
            s_rows.append(row)
            for m in (maps_by.get(r.match_id, pd.DataFrame()).itertuples(index=False)):
                x = eng.map_snapshot(r.team1, r.tier, m.map_name)
                y = eng.map_snapshot(r.team2, r.tier, m.map_name)
                pick = 1 if m.picked_by == r.team1 else (-1 if m.picked_by == r.team2 else 0)
                m_rows.append({"match_id": r.match_id, "map_number": m.map_number, "map_name": m.map_name,
                               "date": date, "pick_t1": pick, "is_decider": int(m.picked_by == "decider"),
                               "t1_map_rating": x["map_rating"], "t2_map_rating": y["map_rating"],
                               "t1_map_games": x["map_games"], "t2_map_games": y["map_games"],
                               "t1_map_win": int(m.score1 > m.score2)})
        # 2) now learn from today's results
        for r in day.itertuples(index=False):
            if r.forfeit:
                continue
            eng.update_series(r)
            for m in maps_by.get(r.match_id, pd.DataFrame()).itertuples(index=False):
                eng.update_map(m)
            for team in (r.team1, r.team2):
                g = players_by.get((r.match_id, team))
                if g is not None:
                    eng.update_players(team, date, g)
        eng.end_day(date)
    return pd.DataFrame(s_rows), pd.DataFrame(m_rows), eng

t0 = time.time()
SERIES_F, MAP_F, ENGINE = build_features(SERIES_C, MAPS_C, PLAYERS_C, PARAMS)
feat_dir = CFG.path("features")
SERIES_F.to_parquet(feat_dir / "series_features.parquet", index=False)
MAP_F.to_parquet(feat_dir / "map_features.parquet", index=False)
joblib.dump({**ENGINE.to_state(), "built_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
             "last_date": str(SERIES_C.date.max().date())}, CFG.path("models") / "engine_state.joblib")
_reloaded = FeatureEngine.from_state(joblib.load(CFG.path("models") / "engine_state.joblib"))
def _differences(a: dict, b: dict) -> list:
    out = []
    for k in a.keys() | b.keys():
        x, y = a.get(k), b.get(k)
        if pd.isna(x) and pd.isna(y):
            continue
        if x is None or y is None or not math.isclose(x, y, rel_tol=1e-12, abs_tol=1e-12):
            out.append(f"{k}: {x} vs {y}")
    return out

_day_after = SERIES_C.date.max() + pd.Timedelta(days=1)
for _r in SERIES_C.tail(50).itertuples():
    for _t in (_r.team1, _r.team2):
        _d = _differences(_reloaded.team_snapshot(_t, _r.tier, _day_after), ENGINE.team_snapshot(_t, _r.tier, _day_after))
        assert not _d, f"saved engine state doesn't reload to the same numbers for team {_t}: {_d}"
print("   engine state saved and verified: it reloads to identical numbers")
log.info("Layer 3 features built: %d series rows, %d map rows in %.1fs", len(SERIES_F), len(MAP_F), time.time() - t0)
print(f"✅ {len(SERIES_F):,} series rows × {SERIES_F.shape[1]} columns and {len(MAP_F):,} map rows "
      f"in {time.time() - t0:.0f} s")
print(f"   Saved to {feat_dir}")

### 3.4 Leakage test — proof that no feature sees the future
**The future-deletion test:** delete every match after a cutoff date, rebuild all features from scratch, and compare. Every feature for matches *before* the cutoff must be **exactly** the same as in the full build. If any value changes, some feature was using information from after the match — and the cell stops with an error.

It runs at three cutoff dates. A second check shuffles the input rows and confirms the result doesn't depend on row order.

In [ ]:
FEATURE_COLS = [c for c in SERIES_F.columns
                if c.startswith(("t1_", "t2_", "h2h_", "elo_p", "days_since")) and c != "t1_win"]

def frames_equal(a: pd.DataFrame, b: pd.DataFrame, cols) -> tuple[bool, str]:
    a, b = a[cols].astype(float), b[cols].astype(float)
    diff = ~(np.isclose(a.values, b.values, rtol=0, atol=1e-9) | (np.isnan(a.values) & np.isnan(b.values)))
    if diff.any():
        r, c = np.argwhere(diff)[0]
        return False, f"column '{cols[c]}' differs for match {a.index[r]} ({a.iat[r, c]} vs {b.iat[r, c]})"
    return True, ""

def future_deletion_test(build, series, maps, players, params, fractions=(0.4, 0.7, 0.9)):
    full = build(series, maps, players, params)[0].set_index("match_id")
    cols = [c for c in full.columns if c in FEATURE_COLS]
    days = np.sort(series.date.unique())
    results = []
    for frac in fractions:
        cutoff = days[int(len(days) * frac)]
        keep = series[series.date < cutoff]
        ids = set(keep.match_id)
        part = build(keep, maps[maps.match_id.isin(ids)], players[players.match_id.isin(ids)], params)[0]
        part = part.set_index("match_id")
        ok, why = frames_equal(full.loc[part.index], part, cols)
        results.append((str(pd.Timestamp(cutoff).date()), len(part), ok, why))
    return results

t0 = time.time()
LEAK_RESULTS = future_deletion_test(build_features, SERIES_C, MAPS_C, PLAYERS_C, PARAMS)
for cutoff, n, ok, why in LEAK_RESULTS:
    print(f"   cutoff {cutoff}: {n:6,} earlier matches compared — " + ("identical ✅" if ok else f"❌ {why}"))

shuffled = build_features(SERIES_C.sample(frac=1, random_state=1), MAPS_C.sample(frac=1, random_state=2),
                          PLAYERS_C.sample(frac=1, random_state=3), PARAMS)[0].set_index("match_id")
SHUFFLE_OK, why = frames_equal(SERIES_F.set_index("match_id").loc[shuffled.index], shuffled, FEATURE_COLS)
print("   shuffled input rows — " + ("identical ✅" if SHUFFLE_OK else f"❌ {why}"))

LEAK_FREE = all(ok for *_, ok, _ in LEAK_RESULTS) and SHUFFLE_OK
(CFG.path("logs") / "layer3_leakage.json").write_text(json.dumps(
    {"cutoffs": LEAK_RESULTS, "shuffle_ok": SHUFFLE_OK, "leak_free": LEAK_FREE}, indent=2, default=str))
log.info("Leakage test: leak_free=%s (%.0fs)", LEAK_FREE, time.time() - t0)
if not LEAK_FREE:
    raise RuntimeError("Leakage test FAILED — a feature can see the future. Send this output to Claude.")
print(f"\n✅ Leakage test passed ({time.time() - t0:.0f} s): no feature uses information from after its match.")

### 3.5 Sanity checks
Not the real evaluation (that's Layer 8, with a proper train/test split) — just a quick look that the features behave sensibly:
- **Elo alone:** how often does the team with the higher pre-match (region-adjusted) Elo win? Good Elo systems in esports usually land somewhere around 60–70%; far below means something's off, suspiciously far above means something leaked.
- **Top teams right now** by Elo — you should recognise these names.
- **Missing values** per feature — expected for roster features in Challengers (no player data there).

In [ ]:
usable = SERIES_F[(SERIES_F.forfeit == 0) & (SERIES_F.t1_n_series >= 5) & (SERIES_F.t2_n_series >= 5)].copy()
usable["elo_pick_right"] = ((usable.t1_elo_adj > usable.t2_elo_adj) == (usable.t1_win == 1)).astype(int)
usable["year"] = usable.date.dt.year
print(f"Higher-Elo team won {usable.elo_pick_right.mean():.1%} of {len(usable):,} series "
      "(both teams with ≥ 5 previous series)")
print(usable.pivot_table(index="tier", columns="year", values="elo_pick_right", aggfunc="mean")
      .map(lambda v: f"{v:.0%}" if pd.notna(v) else "").to_string())

intl = usable[usable.tier == "international"]
raw_right = ((intl.t1_elo > intl.t2_elo) == (intl.t1_win == 1)).mean()
print(f"\nInternational matches: {intl.elo_pick_right.mean():.1%} with region adjustment vs "
      f"{raw_right:.1%} without ({len(intl)} series)")
print("Region strength offsets now: " + ", ".join(f"{k} {v:+.0f}" for k, v in
                                                 sorted(ENGINE.region_offset.items(), key=lambda x: -x[1])))

names = dict(zip(TEAMS_C.key, TEAMS_C["name"]))
cutoff = SERIES_C.date.max() - pd.Timedelta(days=120)
active = {t for t, d in ENGINE.last_date.items() if d >= cutoff}
top = sorted(((ENGINE.elo[t] + ENGINE.offset(t), t) for t in active
              if ENGINE.n_tier1[t] / max(ENGINE.n_series[t], 1) >= 0.5), reverse=True)[:12]
print("\nTop VCT teams by region-adjusted Elo right now (active in the last 120 days):")
for rank, (elo, t) in enumerate(top, 1):
    roster = ENGINE.lineups.get(t)
    rr = np.mean([ENGINE.player_rating(x) for x in sorted(roster[-1][1])]) if roster else float("nan")
    print(f"   {rank:2d}. {names.get(t, t):28s} Elo {elo:6.0f}   {ENGINE.team_region.get(t, '?'):9s} "
          f"roster rating {rr:.2f}")

print("\nMissing values (share of rows):")
miss = SERIES_F[[f"t1_{c}" for c in TEAM_FEATURES]].isna().mean()
for col, share in miss[miss > 0].items():
    by_tier = SERIES_F.groupby("tier")[col].apply(lambda s: s.isna().mean())
    print(f"   {col:22s} {share:5.0%}   " + "  ".join(f"{t}: {v:.0%}" for t, v in by_tier.items()))

LAYER3_OK = LEAK_FREE and 0.55 <= usable.elo_pick_right.mean() <= 0.80
print("\n🎉 Layer 3 complete — ready for Layer 4." if LAYER3_OK else
      "\n⚠️ Elo accuracy is outside the expected 55–80% range — send this output to Claude.")

---
# Layer 4 — Dataset

Turns the feature table into what the models actually learn from.

**1. Features as differences.** The model sees *team 1 minus team 2* for every team number (Elo, form, roster rating…), plus the match context (Bo1/3/5, tier, playoffs, patch age). A model that sees differences can't care which team vlr.gg happened to list first. Elo appears **once**, as the region-adjusted Elo win chance (`elo_logit`) — raw Elo and the region offset are left out because adjusted Elo already contains them, and exact duplicates make a model's weights meaningless.

**2. Mirrored rows.** Every training match is added twice: once as *A vs B* and once as *B vs A* with the result flipped. Every prediction is also made both ways and averaged, so **P(A beats B) + P(B beats A) = 100%, always**.

**3. Time-ordered blocks** — never shuffled. A model must be tested on matches that happened *after* everything it learned from, exactly like predicting a real upcoming match:

| Block | Share | Used for |
|---|---|---|
| train | 65% (oldest) | fitting the models |
| tune | 12% | choosing settings and comparing models (Layer 5) |
| calibrate | 8% | making the percentages honest (Layer 7) |
| **test** | **15% (newest)** | **the final, one-time score (Layer 8)** — nothing touches it before then |

Blocks are cut between days, so no day is split across two blocks. The first `WARMUP_DAYS` (90) are left out of training: every team starts at a default rating, and those early numbers mean little.

### 4.1 Build the feature matrix

In [ ]:
import json, numpy as np, pandas as pd

feat_dir = CFG.path("features")
SF = pd.read_parquet(feat_dir / "series_features.parquet")
MF = pd.read_parquet(feat_dir / "map_features.parquet")
SF["date"] = pd.to_datetime(SF["date"])
MF["date"] = pd.to_datetime(MF["date"])

WARMUP_DAYS = 90
TIER1 = ("international", "regional")

# Team numbers that become "team 1 minus team 2" features.
# Elo enters once, as elo_logit (= region-adjusted Elo difference × ln10/400). Raw Elo, region offset and
# adjusted Elo are left out on purpose: adjusted = raw + offset, so including all three gives the model
# exact duplicates and makes its weights meaningless.
DIFF_FEATURES = ["map_elo", "form", "round_diff", "tier1_share",
                 "roster_rating", "roster_continuity", "roster_days_old", "patch_winrate", "days_rest", "is_new"]

def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

def make_X(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)
    for f in DIFF_FEATURES:
        X[f"d_{f}"] = df[f"t1_{f}"] - df[f"t2_{f}"]
    X["d_log_n"] = np.log1p(df.t1_n_series) - np.log1p(df.t2_n_series)
    X["elo_logit"] = _logit(df.elo_p_t1)
    X["h2h"] = df.h2h_t1 - 0.5
    is_tier1 = df.tier.isin(TIER1).astype(float)
    X["elo_x_bo1"] = X.elo_logit * (df.best_of == 1)          # Bo1s are more random…
    X["elo_x_bo5"] = X.elo_logit * (df.best_of == 5)          # …Bo5s favour the stronger team
    X["elo_x_tier1"] = X.elo_logit * is_tier1
    # context — the same whichever team is listed first
    X["best_of"] = df.best_of.astype(float)
    X["is_playoffs"] = df.is_playoffs.astype(float)
    X["is_international"] = (df.tier == "international").astype(float)
    X["is_regional"] = (df.tier == "regional").astype(float)
    X["days_since_patch"] = df.days_since_patch.astype(float)
    X["min_log_n"] = np.log1p(np.minimum(df.t1_n_series, df.t2_n_series))
    X["h2h_n"] = df.h2h_n.astype(float)
    X["both_roster"] = (df.t1_roster_rating.notna() & df.t2_roster_rating.notna()).astype(float)
    return X

# "Antisymmetric" columns flip sign when the teams swap; "symmetric" ones don't change.
ANTISYM = [f"d_{f}" for f in DIFF_FEATURES] + ["d_log_n", "elo_logit", "h2h", "elo_x_bo1", "elo_x_bo5", "elo_x_tier1"]
SYM = ["best_of", "is_playoffs", "is_international", "is_regional", "days_since_patch", "min_log_n", "h2h_n", "both_roster"]
FEATURES = ANTISYM + SYM

def mirror(X: pd.DataFrame) -> pd.DataFrame:
    """The same matches seen from the other team's side."""
    Y = X.copy()
    Y[[c for c in ANTISYM if c in Y]] = -Y[[c for c in ANTISYM if c in Y]]
    return Y

start = SF.date.min() + pd.Timedelta(days=WARMUP_DAYS)
data = SF[(SF.forfeit == 0) & (SF.date >= start)].sort_values(["date", "match_id"]).reset_index(drop=True)
X = make_X(data)
assert list(X.columns) == FEATURES
assert np.allclose(mirror(mirror(X)).fillna(0).values, X.fillna(0).values), "mirroring twice must give back the original"
print(f"✅ {len(data):,} usable series ({len(SF) - len(data):,} left out: forfeits and the first {WARMUP_DAYS} days)")
print(f"   {len(FEATURES)} features: {len(ANTISYM)} that flip with the team order, {len(SYM)} context")

### 4.2 Split into time-ordered blocks and save

In [ ]:
def assign_blocks(dates: pd.Series) -> pd.Series:
    """Give every day one block, using the share of rows that came before that day."""
    fracs = [CFG.split_train, CFG.split_tune, CFG.split_calibrate]
    bounds = np.cumsum(fracs)
    per_day = dates.value_counts().sort_index()
    before = (per_day.cumsum() - per_day) / per_day.sum()
    names = np.array(["train", "tune", "calibrate", "test"])
    day_block = pd.Series(names[np.searchsorted(bounds, before.values, side="right")], index=per_day.index)
    return dates.map(day_block)

data["block"] = assign_blocks(data.date)
DATASET = pd.concat([data[["match_id", "date", "tier", "team1", "team2", "block", "t1_win"]], X],
                    axis=1).rename(columns={"t1_win": "y"})       # best_of is already one of the features
assert DATASET.columns.is_unique

# Map-level dataset (used in Layer 6): the series features plus how strong each team is on that map.
md_ = MF.merge(DATASET.drop(columns=["date", "tier"]), on="match_id", how="inner")
md_["d_map_rating"] = md_.t1_map_rating - md_.t2_map_rating
md_["d_log_map_games"] = np.log1p(md_.t1_map_games) - np.log1p(md_.t2_map_games)
md_["pick"] = md_.pick_t1.astype(float)
MAP_ANTISYM = ANTISYM + ["d_map_rating", "d_log_map_games", "pick"]
MAP_SYM = SYM + ["is_decider", "map_number"]
MAP_DATASET = md_[["match_id", "map_number", "map_name", "date", "block", "team1", "team2"] +
                  [c for c in MAP_ANTISYM + MAP_SYM if c != "map_number"] + ["t1_map_win"]].rename(columns={"t1_map_win": "y"})

# ---- checks ----
order = ["train", "tune", "calibrate", "test"]
ranges = DATASET.groupby("block").date.agg(["min", "max", "size"]).reindex(order)
for a, b in zip(order, order[1:]):
    assert ranges.loc[a, "max"] < ranges.loc[b, "min"], f"blocks {a} and {b} overlap in time"
assert DATASET.groupby("date").block.nunique().max() == 1, "a day was split across blocks"
assert DATASET.match_id.is_unique

DATASET.to_parquet(feat_dir / "dataset.parquet", index=False)
MAP_DATASET.to_parquet(feat_dir / "map_dataset.parquet", index=False)
meta = {"features": FEATURES, "antisym": ANTISYM, "sym": SYM, "map_antisym": MAP_ANTISYM, "map_sym": MAP_SYM,
        "warmup_days": WARMUP_DAYS,
        "blocks": {b: [str(ranges.loc[b, "min"].date()), str(ranges.loc[b, "max"].date()), int(ranges.loc[b, "size"])]
                   for b in order}}
(feat_dir / "dataset_meta.json").write_text(json.dumps(meta, indent=2))

print("✅ Blocks (oldest → newest), never overlapping in time:")
counts = pd.crosstab(DATASET.block, DATASET.tier).reindex(order)
for b in order:
    lo, hi, n = meta["blocks"][b]
    by_tier = "  ".join(f"{t}: {counts.loc[b, t]:,}" for t in counts.columns)
    print(f"   {b:9s} {lo} → {hi}   {n:6,} series   ({by_tier})")
print(f"\n   team 1 (as listed by vlr.gg) won {DATASET.y.mean():.1%} of series — mirroring removes this bias")
print(f"   map dataset: {len(MAP_DATASET):,} maps")
miss = DATASET[FEATURES].isna().mean()
print("   features with missing values (filled as 'no difference' for Logistic Regression; LightGBM handles them):")
print("      " + ", ".join(f"{c} {v:.0%}" for c, v in miss[miss > 0].sort_values(ascending=False).items()))
log.info("Layer 4 done: %s", meta["blocks"])
print("\n🎉 Layer 4 complete.")

---
# Layer 5 — Models

Several models, from simplest to most flexible. **Each must earn its place by beating the simpler ones on the *tune* block** — matches the models never trained on.

| Model | What it is |
|---|---|
| **Elo** | Just the region-adjusted Elo win chance. No learning at all — the baseline everything must beat. |
| **Elo (calibrated)** | Elo, with one number learned to fix how confident it is (raw Elo is often over- or under-confident). |
| **Logistic Regression** | A weighted sum of all the team-difference features. Simple, hard to overfit. |
| **LightGBM** | Gradient-boosted decision trees: can learn interactions (e.g. form matters more in Bo1s). Constrained so a higher Elo can never *lower* a team's chance. |
| **Ensemble** | The average of Logistic Regression and LightGBM — only chosen if it is *clearly* better. |

**How we score them** (lower is better for the first two):
- **Log loss** — the main score. Punishes confident wrong predictions hard: saying 90% and being wrong costs far more than saying 55% and being wrong. This rewards *honest* percentages, which is what you want from a win-chance predictor.
- **Brier score** — the average squared error of the percentages.
- **Accuracy** — how often the favourite won. Easy to read, but it ignores *how* confident the model was.

**VCT matches count 3× (`CFG.vct_weight`)** — in training and when choosing the model. Challengers is ~85% of the data, so without this the models mostly learn Challengers and end up unsure about VCT, which is what we most want to predict.

The **test block is not used anywhere in this layer** — it's saved for one honest final score in Layer 8.

### 5.1 Training helpers

In [ ]:
import json, time, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss, brier_score_loss
import lightgbm as lgb

feat_dir = CFG.path("features")
DS = pd.read_parquet(feat_dir / "dataset.parquet")
META = json.loads((feat_dir / "dataset_meta.json").read_text())
FEATURES, ANTISYM, SYM = META["features"], META["antisym"], META["sym"]
TRAIN = DS[DS.block == "train"].reset_index(drop=True)
TUNE = DS[DS.block == "tune"].reset_index(drop=True)
# The calibrate and test blocks are deliberately not loaded here.

def mirror_X(X: pd.DataFrame) -> pd.DataFrame:
    Y = X.copy()
    cols = [c for c in ANTISYM if c in Y.columns]
    Y[cols] = -Y[cols]
    return Y

def mirrored(df: pd.DataFrame, cols):
    """Training data plus its mirror image (teams swapped, result flipped), and the matching weights."""
    X = df[cols]
    w = match_weights(df)
    return (pd.concat([X, mirror_X(X)], ignore_index=True), np.concatenate([df.y.values, 1 - df.y.values]),
            np.concatenate([w, w]))

TIER1 = ("international", "regional")
def match_weights(df: pd.DataFrame) -> np.ndarray:
    return np.where(df.tier.isin(TIER1), CFG.vct_weight, 1.0)

def symmetric(predict, X: pd.DataFrame) -> np.ndarray:
    """Average of 'A vs B' and '1 − (B vs A)': guarantees P(A wins) + P(B wins) = 1."""
    return 0.5 * (predict(X) + 1 - predict(mirror_X(X)))

def scores(y, p, w=None) -> dict:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"log_loss": log_loss(y, p, labels=[0, 1], sample_weight=w),
            "brier": brier_score_loss(y, p, sample_weight=w),
            "accuracy": float(np.average((p > 0.5) == (y == 1), weights=w))}

def paired_bootstrap(y, p_a, p_b, w=None, n=CFG.bootstrap_samples, seed=CFG.seed):
    """How much better is B than A in (weighted) log loss, with a 95% range (negative = B is better)."""
    rng = np.random.default_rng(seed)
    w = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    la = -(y * np.log(np.clip(p_a, 1e-6, 1)) + (1 - y) * np.log(np.clip(1 - p_a, 1e-6, 1)))
    lb = -(y * np.log(np.clip(p_b, 1e-6, 1)) + (1 - y) * np.log(np.clip(1 - p_b, 1e-6, 1)))
    d = lb - la
    idx = rng.integers(0, len(d), size=(n, len(d)))
    means = (d[idx] * w[idx]).sum(axis=1) / w[idx].sum(axis=1)
    return float(np.average(d, weights=w)), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

print(f"✅ train: {len(TRAIN):,} series ({TRAIN.date.min().date()} → {TRAIN.date.max().date()}),  "
      f"tune: {len(TUNE):,} series ({TUNE.date.min().date()} → {TUNE.date.max().date()})")

### 5.2 Train the models
Settings for Logistic Regression and LightGBM are chosen by trying a small grid and keeping whatever scores best on the tune block. LightGBM also stops adding trees as soon as the tune score stops improving (*early stopping*) — the main guard against overfitting. Takes 1–3 minutes.

In [ ]:
t0 = time.time()
y_tune = TUNE.y.values
w_tune = match_weights(TUNE)
P = {}                                   # model name → tune-block predictions
MODELS = {}

# --- Elo, raw and calibrated ---
P["Elo"] = 1 / (1 + np.exp(-TUNE.elo_logit.values))
Xe, ye, we = mirrored(TRAIN, ["elo_logit"])
elo_cal = LogisticRegression(fit_intercept=False, C=1e6).fit(Xe, ye, sample_weight=we)
MODELS["Elo (calibrated)"] = elo_cal
P["Elo (calibrated)"] = symmetric(lambda X: elo_cal.predict_proba(X[["elo_logit"]])[:, 1], TUNE[["elo_logit"]])

# --- Logistic Regression on the team-difference features ---
# No intercept and only sign-flipping features → swapping the teams exactly flips the prediction.
Xl, yl, wl = mirrored(TRAIN, ANTISYM)
best = None
for C in (0.003, 0.01, 0.03, 0.1, 0.3, 1.0):
    lr = make_pipeline(SimpleImputer(strategy="constant", fill_value=0.0), StandardScaler(),
                       LogisticRegression(C=C, fit_intercept=False, max_iter=2000)).fit(
        Xl, yl, logisticregression__sample_weight=wl)
    p = symmetric(lambda X: lr.predict_proba(X[ANTISYM])[:, 1], TUNE[ANTISYM])
    ll = scores(y_tune, p, w_tune)["log_loss"]
    if best is None or ll < best[0]:
        best = (ll, C, lr, p)
LR_C = best[1]
MODELS["Logistic Regression"], P["Logistic Regression"] = best[2], best[3]

# --- LightGBM ---
Xg, yg, wg = mirrored(TRAIN, FEATURES)
mono = [1 if c == "elo_logit" else 0 for c in FEATURES]
best = None
for leaves in (7, 15):
    for min_child in (50, 200):
        for l2 in (0.0, 5.0):
            m = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=leaves,
                                   min_child_samples=min_child, reg_lambda=l2, subsample=0.8, subsample_freq=1,
                                   colsample_bytree=0.8, monotone_constraints=mono, random_state=CFG.seed,
                                   verbose=-1)
            m.fit(Xg, yg, sample_weight=wg, eval_set=[(TUNE[FEATURES], y_tune)], eval_sample_weight=[w_tune],
                  eval_metric="binary_logloss", callbacks=[lgb.early_stopping(100, verbose=False)])
            p = symmetric(lambda X: m.predict_proba(X[FEATURES])[:, 1], TUNE[FEATURES])
            ll = scores(y_tune, p, w_tune)["log_loss"]
            if best is None or ll < best[0]:
                best = (ll, dict(num_leaves=leaves, min_child_samples=min_child, reg_lambda=l2,
                                 n_estimators=m.best_iteration_), m, p)
LGBM_PARAMS = best[1]
MODELS["LightGBM"], P["LightGBM"] = best[2], best[3]

P["Ensemble"] = 0.5 * (P["Logistic Regression"] + P["LightGBM"])
print(f"✅ Trained in {time.time() - t0:.0f} s.  Logistic Regression C={LR_C}  |  LightGBM {LGBM_PARAMS}")

### 5.3 Compare on the tune block, and choose
The table shows each model's scores on the tune block: the **weighted log loss** (VCT ×3 — this is what decides), plus plain scores for **all** series and for **VCT** only.

The last columns compare each model against **calibrated Elo** with a *paired bootstrap*: the tune matches are re-sampled 1,000 times to see how much the difference in log loss could be down to luck. If the whole 95% range is below zero, the model is genuinely better; if it crosses zero, the difference could be chance.

**Choosing the model:** the simplest model whose tune log loss is within noise of the best is kept — a more complex model has to *clearly* win to be chosen.

In [ ]:
tier1_mask = TUNE.tier.isin(["international", "regional"]).values
rows = []
for name, p in P.items():
    a, v = scores(y_tune, p), scores(y_tune[tier1_mask], p[tier1_mask])
    wll = scores(y_tune, p, w_tune)["log_loss"]
    d, lo, hi = (paired_bootstrap(y_tune, P["Elo (calibrated)"], p, w_tune) if name != "Elo (calibrated)"
                 else (0, 0, 0))
    rows.append({"model": name, "weighted LL": wll, "log loss": a["log_loss"], "brier": a["brier"],
                 "accuracy": a["accuracy"],
                 "VCT log loss": v["log_loss"], "VCT accuracy": v["accuracy"],
                 "vs Elo-cal": d, "95% range": f"[{lo:+.4f}, {hi:+.4f}]"})
RESULTS = pd.DataFrame(rows).set_index("model")
print(RESULTS.to_string(formatters={"weighted LL": "{:.4f}".format, "log loss": "{:.4f}".format,
                                    "brier": "{:.4f}".format,
                                    "accuracy": "{:.1%}".format, "VCT log loss": "{:.4f}".format,
                                    "VCT accuracy": "{:.1%}".format, "vs Elo-cal": "{:+.4f}".format}))
print(f"   ({len(TUNE):,} tune series, {tier1_mask.sum():,} of them VCT)")

# Simplest model within noise of the best wins.
complexity = ["Elo (calibrated)", "Logistic Regression", "LightGBM", "Ensemble"]
best_name = RESULTS.loc[complexity, "weighted LL"].idxmin()
CHOSEN = best_name
for name in complexity:
    if name == best_name:
        break
    d, lo, hi = paired_bootstrap(y_tune, P[name], P[best_name], w_tune)
    if hi >= 0:                      # the best model is not clearly better than this simpler one
        CHOSEN = name
        break
print(f"\n➡️  Chosen model: {CHOSEN}" + ("" if CHOSEN == best_name else
      f"  (best score was {best_name}, but not clearly better than the simpler {CHOSEN})"))

# What the models rely on
imp = pd.Series(MODELS["LightGBM"].booster_.feature_importance("gain"), index=FEATURES)
imp = (imp / imp.sum()).sort_values(ascending=False)
print("\nLightGBM — most useful features (share of total gain):")
print("   " + ", ".join(f"{c} {v:.0%}" for c, v in imp.head(10).items()))
coefs = pd.Series(MODELS["Logistic Regression"][-1].coef_[0], index=ANTISYM).sort_values(key=abs, ascending=False)
print("Logistic Regression — largest weights (on standardised features):")
print("   " + ", ".join(f"{c} {v:+.2f}" for c, v in coefs.head(8).items()))

joblib.dump({"models": MODELS, "chosen": CHOSEN, "features": FEATURES, "antisym": ANTISYM, "sym": SYM,
             "lr_C": LR_C, "lgbm_params": LGBM_PARAMS, "tune_results": RESULTS.reset_index().to_dict("records"),
             "trained_on": META["blocks"]["train"]}, CFG.path("models") / "series_models.joblib")
(CFG.path("logs") / "layer5_results.json").write_text(RESULTS.reset_index().to_json(orient="records", indent=2))
log.info("Layer 5: chosen=%s results=%s", CHOSEN, RESULTS["weighted LL"].round(4).to_dict())
beats_elo = RESULTS.loc[CHOSEN, "weighted LL"] <= RESULTS.loc["Elo", "weighted LL"]
print("\n🎉 Layer 5 complete — models saved." if beats_elo else
      "\n⚠️ The chosen model doesn't beat raw Elo on the tune block — send this output to Claude.")

---
# Layer 6 — Maps and the series breakdown

A Valorant series is won map by map, and teams have favourite and hated maps. This layer adds:

1. **A map model** — the chance each team wins a *given* map, using its strength on that map and who picked it.
2. **Map pool and veto habits** — which maps are currently played, and how often each team picks or bans each one (recent vetoes count more).
3. **A veto simulation** — plays out the pick/ban phase 2,000 times using those habits, giving *which maps will probably be played* and the **series chance built map by map**.

**How the numbers are used:** the headline win chance shown in the app comes from the series model (Layer 5, calibrated in Layer 7, tested in Layer 8). The map breakdown explains it — likely maps and each team's chance on them. The two numbers usually agree within a few percent; a big gap is worth a closer look (e.g. one team's best map is likely to be banned).

### 6.1 Map model
Trained on the train block's maps, compared on the tune block's maps against a simple baseline: the map-rating difference alone (Elo applied to that map).

In [ ]:
import json, math, numpy as np, pandas as pd, joblib
from collections import Counter, defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss

feat_dir, clean_dir = CFG.path("features"), CFG.path("clean")
META = json.loads((feat_dir / "dataset_meta.json").read_text())
MAP_ANTI = META["map_antisym"]
MD = pd.read_parquet(feat_dir / "map_dataset.parquet")

def mirror_cols(X: pd.DataFrame, anti) -> pd.DataFrame:
    Y = X.copy()
    cols = [c for c in anti if c in Y.columns]
    Y[cols] = -Y[cols]
    return Y

def sym_predict(predict, X: pd.DataFrame, anti) -> np.ndarray:
    return 0.5 * (predict(X) + 1 - predict(mirror_cols(X, anti)))

m_train, m_tune = MD[MD.block == "train"], MD[MD.block == "tune"]
Xm = pd.concat([m_train[MAP_ANTI], mirror_cols(m_train[MAP_ANTI], MAP_ANTI)], ignore_index=True)
ym = np.concatenate([m_train.y.values, 1 - m_train.y.values])
best = None
for C in (0.003, 0.01, 0.03, 0.1, 0.3, 1.0):
    mm = make_pipeline(SimpleImputer(strategy="constant", fill_value=0.0), StandardScaler(),
                       LogisticRegression(C=C, fit_intercept=False, max_iter=2000)).fit(Xm, ym)
    p = sym_predict(lambda X: mm.predict_proba(X[MAP_ANTI])[:, 1], m_tune[MAP_ANTI], MAP_ANTI)
    ll = log_loss(m_tune.y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1])
    if best is None or ll < best[0]:
        best = (ll, C, mm, p)
MAP_LL, MAP_C, MAP_MODEL, p_map = best
p_base = 1 / (1 + 10 ** (-m_tune.d_map_rating.values / 400))
base_ll = log_loss(m_tune.y, np.clip(p_base, 1e-6, 1 - 1e-6), labels=[0, 1])
acc = lambda p: np.mean((p > 0.5) == (m_tune.y.values == 1))
print(f"Tune block: {len(m_tune):,} maps")
print(f"   map rating only : log loss {base_ll:.4f}   accuracy {acc(p_base):.1%}")
print(f"   map model       : log loss {MAP_LL:.4f}   accuracy {acc(p_map):.1%}   (C={MAP_C})")
coef = pd.Series(MAP_MODEL[-1].coef_[0], index=MAP_ANTI)
print(f"   effect of picking the map yourself (standardised weight): {coef['pick']:+.3f}")

### 6.2 Map pool and veto habits
- **Map pool** = maps played in the last 120 days (the 7 most played) — Riot rotates maps in and out, so this changes over time.
- **Veto habits** = for each team, how often it bans or picks each map, with a half-life of 180 days so recent habits count most.

In [ ]:
SERIES_CLEAN = pd.read_parquet(clean_dir / "series.parquet")[["match_id", "date"]]
SERIES_CLEAN["date"] = pd.to_datetime(SERIES_CLEAN["date"])
VETO = pd.read_parquet(clean_dir / "vetoes.parquet").merge(SERIES_CLEAN, on="match_id", how="inner")
MAPS_ALL = pd.read_parquet(clean_dir / "maps.parquet")
MAPS_ALL["date"] = pd.to_datetime(MAPS_ALL["date"])

def map_pool(as_of, days=120, size=7):
    recent = MAPS_ALL[(MAPS_ALL.date < as_of) & (MAPS_ALL.date >= as_of - pd.Timedelta(days=days))]
    return list(recent.map_name.value_counts().head(size).index)

def veto_profile(as_of, halflife_days=180) -> dict:
    """(team, 'ban'/'pick', map) → time-weighted count, using only vetoes before `as_of`."""
    v = VETO[(VETO.date < as_of) & VETO.team.notna() & VETO.action.isin(["ban", "pick"])]
    w = 0.5 ** ((as_of - v.date).dt.days / halflife_days)
    return v.assign(w=w).groupby(["team", "action", "map_name"]).w.sum().to_dict()

NOW = SERIES_CLEAN.date.max() + pd.Timedelta(days=1)
POOL_NOW = map_pool(NOW)
PROFILE_NOW = veto_profile(NOW)
print(f"✅ Current map pool ({len(POOL_NOW)}): {', '.join(POOL_NOW)}")
print(f"   veto habits known for {len({k[0] for k in PROFILE_NOW})} teams")

### 6.3 Veto simulation and the map-by-map breakdown
The veto order follows the standard VCT format, built from the pool size:

| Best of | Order (7-map pool) |
|---|---|
| Bo1 | ban ×6 alternating → the last map is played |
| Bo3 | ban, ban, **pick, pick**, ban, ban → decider |
| Bo5 | ban, ban, **pick, pick, pick, pick** → decider |

At each step the acting team chooses using its habits (how often it bans/picks that map), nudged by strength: teams lean towards picking maps where they're stronger than the opponent and banning maps where they're weaker. Who starts the veto is random (50/50), since it depends on seeding we don't know.

The series chance from a list of maps is exact arithmetic: e.g. a 60% chance on every map gives **64.8%** for a Bo3 and **68.3%** for a Bo5 — longer series favour the stronger team.

In [ ]:
from functools import lru_cache

def veto_sequence(best_of: int, pool_size: int) -> list:
    picks, bans = best_of - 1, pool_size - best_of
    first = min(2, bans)
    return ["ban"] * first + ["pick"] * picks + ["ban"] * (bans - first)

def series_chance(map_probs: list, best_of: int) -> float:
    """Exact chance team 1 wins the series, given its chance on each map in play order."""
    need = best_of // 2 + 1
    @lru_cache(maxsize=None)
    def go(i, a, b):
        if a == need: return 1.0
        if b == need: return 0.0
        p = map_probs[i]
        return p * go(i + 1, a + 1, b) + (1 - p) * go(i + 1, a, b + 1)
    return go(0, 0, 0)

def match_features(eng, t1, t2, tier="regional", best_of=3, date=None, is_playoffs=0) -> pd.DataFrame:
    """One feature row for a match that hasn't happened, from the engine's current state."""
    date = date or NOW
    a, b = eng.team_snapshot(t1, tier, date), eng.team_snapshot(t2, tier, date)
    row = {"match_id": "upcoming", "date": date, "tier": tier, "best_of": best_of, "team1": t1, "team2": t2,
           "is_playoffs": int(is_playoffs), "days_since_patch": eng.days_since_patch(date), "forfeit": 0,
           "has_detail": 1, "t1_win": 0}
    row.update({f"t1_{k}": v for k, v in a.items()})
    row.update({f"t2_{k}": v for k, v in b.items()})
    row.update(eng.pair_snapshot(t1, t2, date))
    row["elo_p_t1"] = 1 / (1 + 10 ** ((b["elo_adj"] - a["elo_adj"]) / 400))
    return make_X(pd.DataFrame([row]))

def map_win_chances(eng, X_series: pd.DataFrame, t1, t2, tier, pool) -> dict:
    """map → {pick: chance team 1 wins it} for pick = +1 (t1 picked), −1 (t2 picked), 0 (decider)."""
    out = {}
    for m in pool:
        x, y = eng.map_snapshot(t1, tier, m), eng.map_snapshot(t2, tier, m)
        rows = []
        for pick in (1, -1, 0):
            r = X_series.iloc[0].to_dict()
            r.update({"d_map_rating": x["map_rating"] - y["map_rating"],
                      "d_log_map_games": math.log1p(x["map_games"]) - math.log1p(y["map_games"]),
                      "pick": float(pick), "is_decider": float(pick == 0), "map_number": 1.0})
            rows.append(r)
        X = pd.DataFrame(rows)
        p = sym_predict(lambda Z: MAP_MODEL.predict_proba(Z[MAP_ANTI])[:, 1], X, MAP_ANTI)
        out[m] = {1: p[0], -1: p[1], 0: p[2]}
    return out

def simulate_series(eng, t1, t2, tier="regional", best_of=3, pool=None, profile=None,
                    n=CFG.veto_simulations, seed=CFG.seed) -> dict:
    pool = list(pool or POOL_NOW)
    profile = PROFILE_NOW if profile is None else profile
    if len(pool) < best_of:
        raise ValueError(f"map pool has {len(pool)} maps — too few for a Bo{best_of}")
    X = match_features(eng, t1, t2, tier, best_of)
    chances = map_win_chances(eng, X, t1, t2, tier, pool)
    adv = {m: eng.map_snapshot(t1, tier, m)["map_rating"] - eng.map_snapshot(t2, tier, m)["map_rating"] for m in pool}
    seq = veto_sequence(best_of, len(pool))
    rng = np.random.default_rng(seed)
    played_count, total = Counter(), 0.0
    for s in range(n):
        first = s % 2                                   # exactly half the simulations each team starts
        avail, played = list(pool), []
        for i, action in enumerate(seq):
            me = t1 if (first + i) % 2 == 0 else t2
            sign = 1 if me == t1 else -1
            w = np.array([(profile.get((me, action, m), 0.0) + 0.5) *
                          math.exp((sign * adv[m] if action == "pick" else -sign * adv[m]) / 200) for m in avail])
            m = avail.pop(int(rng.choice(len(avail), p=w / w.sum())))
            if action == "pick":
                played.append((m, sign))
        played.append((avail[0], 0))                     # the decider
        probs = [chances[m][pk] for m, pk in played]
        total += series_chance(tuple(probs), best_of)
        played_count.update(m for m, _ in played)
    maps = [{"map": m, "play_chance": played_count[m] / n,
             "t1_chance_if_t1_picks": chances[m][1], "t1_chance_if_t2_picks": chances[m][-1],
             "t1_chance_as_decider": chances[m][0]} for m in pool]
    maps.sort(key=lambda r: -r["play_chance"])
    return {"series_from_maps": total / n, "maps": maps}

assert abs(series_chance((0.6, 0.6, 0.6), 3) - 0.648) < 1e-12
assert abs(series_chance((0.6,) * 5, 5) - 0.68256) < 1e-12
assert veto_sequence(3, 7) == ["ban", "ban", "pick", "pick", "ban", "ban"]
assert veto_sequence(5, 7) == ["ban", "ban", "pick", "pick", "pick", "pick"]
assert veto_sequence(1, 7) == ["ban"] * 6
joblib.dump({"map_model": MAP_MODEL, "map_antisym": MAP_ANTI, "map_C": MAP_C,
             "tune_log_loss": {"map_rating": base_ll, "map_model": MAP_LL}}, CFG.path("models") / "map_model.joblib")
(CFG.path("logs") / "layer6_results.json").write_text(json.dumps(
    {"tune_maps": len(m_tune), "map_rating_ll": base_ll, "map_model_ll": MAP_LL, "map_C": MAP_C}, indent=2))
print("✅ Veto simulation ready (series arithmetic checked: 60% per map → 64.8% Bo3, 68.3% Bo5).")

---
# Layer 7 — Calibration and the predictor

A model can pick winners well and still give **dishonest percentages** — e.g. say "80%" for matches the favourite only wins 65% of the time. For a win-chance predictor, honest percentages matter as much as picking right. **Calibration** fixes this.

- It's learned on the **calibrate block**: newer than anything the models trained on, and still untouched by the final test.
- Method: *Platt scaling* — one number that stretches or shrinks the model's confidence, plus a second one just for VCT matches.
- **Cautious by design:** with only a few hundred VCT matches in the calibrate block, a strong correction could just be noise — and a noisy ×2.7 stretch amplifies every quirk of the model. So the fit is held towards "no change" (`CFG.calibration_prior`) and limited to between ×0.33 and ×3. A big correction only happens when the data clearly supports it.
- It keeps the symmetry: P(A wins) + P(B wins) is still exactly 100%.

Then everything is put together into **`predict_match()`** — give it two team names, get the win chance, the map breakdown, and notes on how much to trust it.

### 7.1 Fit the calibration
The **reliability table** groups predictions by how confident they were and compares that with how often the favourite actually won. Perfect calibration = the two columns match. (These numbers are on the calibrate block itself, so they flatter the result a little — the honest check is Layer 8.)

In [ ]:
import json, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from scipy.optimize import minimize

DS = pd.read_parquet(CFG.path("features") / "dataset.parquet")
CAL_BLOCK = DS[DS.block == "calibrate"].reset_index(drop=True)
ART = joblib.load(CFG.path("models") / "series_models.joblib")
TIER1 = ("international", "regional")

def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def base_predict(X: pd.DataFrame) -> np.ndarray:
    """The model chosen in Layer 5, with both-ways averaging."""
    M, anti, feats = ART["models"], ART["antisym"], ART["features"]
    fns = {"Logistic Regression": lambda Z: M["Logistic Regression"].predict_proba(Z[anti])[:, 1],
           "LightGBM": lambda Z: M["LightGBM"].predict_proba(Z[feats])[:, 1],
           "Elo (calibrated)": lambda Z: M["Elo (calibrated)"].predict_proba(Z[["elo_logit"]])[:, 1]}
    sym = lambda f: 0.5 * (f(X) + 1 - f(mirror_cols(X, anti)))
    if ART["chosen"] == "Ensemble":
        return 0.5 * (sym(fns["Logistic Regression"]) + sym(fns["LightGBM"]))
    return sym(fns[ART["chosen"]])

def fit_calibrator(p, y, is_tier1, prior=CFG.calibration_prior, bounds=(0.33, 3.0)):
    """Find multipliers a (all matches) and a+b (VCT) for the model's confidence.
    The penalty prior·((a−1)² + b²) keeps them near 1 unless the data clearly says otherwise."""
    z, t, y = _logit(p), np.asarray(is_tier1, dtype=float), np.asarray(y, dtype=float)
    def loss(theta):
        a, b = theta
        s = a * z + b * z * t
        return np.sum(np.logaddexp(0, s) - y * s) + prior * ((a - 1) ** 2 + b ** 2)
    lo, hi = bounds
    a, b = minimize(loss, x0=[1.0, 0.0], method="L-BFGS-B", bounds=[(lo, hi), (lo - hi, hi - lo)]).x
    b = float(np.clip(a + b, lo, hi) - a)              # the VCT multiplier stays inside the limits too
    return float(a), b

def apply_calibration(p, is_tier1, a, b):
    z = _logit(p)
    return 1 / (1 + np.exp(-(a * z + b * z * np.asarray(is_tier1, dtype=float))))

def reliability(p, y, bins=(0.5, 0.55, 0.6, 0.65, 0.7, 0.8, 1.0001)) -> pd.DataFrame:
    fav = np.maximum(p, 1 - p)
    fav_won = np.where(p >= 0.5, y == 1, y == 0)
    cut = pd.cut(fav, bins, right=False)
    t = pd.DataFrame({"bin": cut, "pred": fav, "won": fav_won}).groupby("bin", observed=True).agg(
        matches=("won", "size"), predicted=("pred", "mean"), actual=("won", "mean"))
    return t

def ece(p, y) -> float:
    t = reliability(p, y)
    return float((t.matches * (t.predicted - t.actual).abs()).sum() / t.matches.sum())

y_cal = CAL_BLOCK.y.values
is_t1 = CAL_BLOCK.tier.isin(TIER1).values.astype(float)
p_raw = base_predict(CAL_BLOCK[ART["features"]])
CAL_A, CAL_B = fit_calibrator(p_raw, y_cal, is_t1)
p_cal = apply_calibration(p_raw, is_t1, CAL_A, CAL_B)

print(f"Calibrate block: {len(CAL_BLOCK):,} series ({int(is_t1.sum())} VCT).  Model being calibrated: {ART['chosen']}")
print(f"   confidence multiplier: Challengers ×{CAL_A:.2f}, VCT ×{CAL_A + CAL_B:.2f}   "
      "(below 1 = the model was over-confident, above 1 = under-confident)")
ll = lambda p, m=slice(None): log_loss(y_cal[m], np.clip(p[m], 1e-6, 1 - 1e-6), labels=[0, 1])
vct = is_t1 == 1
print(f"   log loss before → after: all {ll(p_raw):.4f} → {ll(p_cal):.4f}   "
      f"VCT {ll(p_raw, vct):.4f} → {ll(p_cal, vct):.4f}")
print(f"   average calibration error (ECE): {ece(p_raw, y_cal):.3f} → {ece(p_cal, y_cal):.3f}")
r_before, r_after = reliability(p_raw, y_cal), reliability(p_cal, y_cal)
print("\nReliability (favourite's predicted chance vs how often favourites actually won):")
print(r_before.join(r_after, lsuffix=" before", rsuffix=" after").to_string(
    formatters={c: "{:.0%}".format for c in ["predicted before", "actual before", "predicted after", "actual after"]}))

### 7.2 Confidence notes
Every prediction comes with notes on how much to trust it:
- a team has played **fewer than 10** recorded matches → *low confidence*;
- a team's **roster changed recently** (under 60% of its latest lineup played its recent matches) → *low confidence* — ratings lag behind roster changes, which is this model's biggest weakness;
- **no player data** for a team (typical in Challengers) → noted, not penalised.

In [ ]:
def confidence_notes(snap1: dict, snap2: dict, name1: str, name2: str):
    notes, low = [], False
    for s, name in ((snap1, name1), (snap2, name2)):
        if s["n_series"] < 10:
            notes.append(f"{name} has only {s['n_series']} recorded matches"); low = True
        rc = s.get("roster_continuity")
        if rc is not None and not pd.isna(rc) and rc < 0.6:
            notes.append(f"{name}'s roster changed recently ({rc:.0%} of the lineup is from recent matches)"); low = True
        if pd.isna(s.get("roster_rating")):
            notes.append(f"no player data for {name}")
    return ("low" if low else "normal"), notes
print("✅ Confidence rules ready.")

### 7.3 The predictor
`predict_match("Team A", "Team B", best_of=3)` — team names are matched loosely (case, accents, old names all work). It uses the feature engine's state after the **latest** match on record, so predictions reflect current form.

In [ ]:
ENGINE_LIVE = FeatureEngine.from_state(joblib.load(CFG.path("models") / "engine_state.joblib"))
TEAMS_T = pd.read_parquet(CFG.path("clean") / "teams.parquet")
TEAMS_T["last_seen"] = pd.to_datetime(TEAMS_T["last_seen"])
import unicodedata, re as _re
def _norm(s: str) -> str:                                  # same rule as Layer 2's norm_name
    s = unicodedata.normalize("NFKD", s or "")
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return _re.sub(r"\s+", " ", _re.sub(r"[^\w\s]", " ", s)).strip()
_alias_rows = [(_norm(a), r.key, r.last_seen) for r in TEAMS_T.itertuples()
               for a in {str(r.name), *str(r.aliases).split("|")} if a and a != "nan"]

def find_team(name: str) -> str:
    n = _norm(name)
    hits = [(last, key) for alias, key, last in _alias_rows if alias == n]
    if not hits:
        hits = [(last, key) for alias, key, last in _alias_rows if n and n in alias]
    if not hits:
        raise ValueError(f"No team called '{name}' in the data.")
    return max(hits)[1]                                   # most recently active if several match

def team_name(key: str) -> str:
    row = TEAMS_T[TEAMS_T.key == key]
    return row["name"].iloc[0] if len(row) else key

def predict_match(team1: str, team2: str, best_of: int = 3, tier: str = "regional",
                  is_playoffs: bool = False, with_maps: bool = True) -> dict:
    k1, k2 = find_team(team1), find_team(team2)
    if k1 == k2:
        raise ValueError("Pick two different teams.")
    X = match_features(ENGINE_LIVE, k1, k2, tier, best_of, NOW, int(is_playoffs))
    p = float(apply_calibration(base_predict(X), [tier in TIER1], CAL_A, CAL_B)[0])
    s1, s2 = ENGINE_LIVE.team_snapshot(k1, tier, NOW), ENGINE_LIVE.team_snapshot(k2, tier, NOW)
    level, notes = confidence_notes(s1, s2, team_name(k1), team_name(k2))
    out = {"team1": team_name(k1), "team2": team_name(k2), "best_of": best_of, "p_team1": p, "p_team2": 1 - p,
           "confidence": level, "notes": notes, "elo": (s1["elo_adj"], s2["elo_adj"]),
           "data_up_to": str((NOW - pd.Timedelta(days=1)).date())}
    if with_maps:
        out["breakdown"] = b = simulate_series(ENGINE_LIVE, k1, k2, tier, best_of)
        gap = abs(p - b["series_from_maps"])
        if gap > 0.15:
            out["notes"].append(f"the overall model and the map-by-map estimate disagree by {gap:.0%} — "
                                "treat this prediction as uncertain")
            out["confidence"] = "low"
    return out

FRIENDLY = {"elo_logit": "Elo (region-adjusted)", "d_form": "recent form", "h2h": "head-to-head",
            "d_days_rest": "days of rest", "d_roster_rating": "roster rating", "d_tier1_share": "VCT experience",
            "d_patch_winrate": "record on current patch", "d_log_n": "matches played", "d_map_elo": "map Elo",
            "d_round_diff": "round difference", "d_roster_continuity": "roster stability",
            "d_roster_days_old": "age of roster data", "d_is_new": "new team", "elo_x_bo1": "Elo × Bo1",
            "elo_x_bo5": "Elo × Bo5", "elo_x_tier1": "Elo × VCT match"}

def explain_match(team1: str, team2: str, best_of: int = 3, tier: str = "regional", top: int = 8) -> pd.DataFrame:
    """Which features push the prediction towards team 1, from the Logistic Regression model.
    (Exact for Logistic Regression; a close guide if LightGBM or the ensemble is the chosen model.)"""
    k1, k2 = find_team(team1), find_team(team2)
    X = match_features(ENGINE_LIVE, k1, k2, tier, best_of, NOW)
    lr, anti = ART["models"]["Logistic Regression"], ART["antisym"]
    z = lr[1].transform(lr[0].transform(X[anti]))[0]
    contrib = pd.Series(z * lr[-1].coef_[0], index=anti)
    t = pd.DataFrame({"feature": [FRIENDLY.get(c, c) for c in anti], "difference": X[anti].iloc[0].values,
                      f"push toward {team_name(k1)}": contrib.values}, index=anti)
    t = t.reindex(contrib.abs().sort_values(ascending=False).index).head(top)
    return t.reset_index(drop=True)

def show_prediction(r: dict):
    print(f"{r['team1']}  {r['p_team1']:.0%}  —  {r['p_team2']:.0%}  {r['team2']}    (Bo{r['best_of']}, "
          f"confidence: {r['confidence']}, data up to {r['data_up_to']})")
    for n in r["notes"]:
        print(f"   • {n}")
    if "breakdown" in r:
        b = r["breakdown"]
        print(f"   built map by map: {r['team1']} {b['series_from_maps']:.0%}")
        print(f"   {'map':10s} {'played':>7s}   {r['team1'][:14]:>14s} wins it (their pick / opponent's pick / decider)")
        for m in b["maps"]:
            print(f"   {m['map']:10s} {m['play_chance']:7.0%}   {m['t1_chance_if_t1_picks']:>10.0%} / "
                  f"{m['t1_chance_if_t2_picks']:.0%} / {m['t1_chance_as_decider']:.0%}")

joblib.dump({"calibration": {"a": CAL_A, "b": CAL_B}, "chosen": ART["chosen"],
             "data_up_to": str((NOW - pd.Timedelta(days=1)).date())}, CFG.path("models") / "calibration.joblib")
(CFG.path("logs") / "layer7_results.json").write_text(json.dumps(
    {"a": CAL_A, "b": CAL_B, "ll_before": ll(p_raw), "ll_after": ll(p_cal),
     "ece_before": ece(p_raw, y_cal), "ece_after": ece(p_cal, y_cal)}, indent=2))

# Example: the two highest-rated active VCT teams
active = [t for t, d in ENGINE_LIVE.last_date.items() if d >= NOW - pd.Timedelta(days=120)
          and ENGINE_LIVE.n_tier1[t] / max(ENGINE_LIVE.n_series[t], 1) >= 0.5]
top2 = sorted(active, key=lambda t: -(ENGINE_LIVE.elo[t] + ENGINE_LIVE.offset(t)))[:2]
if len(top2) == 2:
    print("Example prediction — the two highest-rated VCT teams right now:\n")
    show_prediction(predict_match(team_name(top2[0]), team_name(top2[1]), best_of=5, tier="international"))
log.info("Layer 7 done: calibration a=%.3f b=%.3f", CAL_A, CAL_B)
print("\n🎉 Layer 7 complete — in a new cell try:  show_prediction(predict_match('Team A', 'Team B'))"
      "  and  explain_match('Team A', 'Team B')")

---
# Layer 8 — The final, honest score

This is the moment the **test block** is opened: the newest 15% of matches, which no model, setting or calibration has ever seen. Everything before this layer was decided without it, so the score here is what you can expect on future matches.

**What's measured**
- The **final predictor** (the chosen model + calibration) against plain **Elo**, the baseline to beat.
- Every score comes with a **95% range** from bootstrapping (re-sampling the test matches 1,000 times) — with a few hundred VCT matches, luck alone moves accuracy by several points, and the range shows how much.
- A **paired** comparison against Elo on the very same matches: is the model *genuinely* better, or could the gap be luck?
- **Calibration** on new data: when it says 70%, does the favourite win about 70% of the time?
- **Stability** month by month — a model that only works in some months isn't reliable.

**One rule:** the test block only means something if you *don't* go back and tweak things until it looks good. Any change made after seeing these numbers needs a fresh test period (new matches) to be judged fairly. The notebook records when the test block was first opened.

### 8.1 Predict every test match

In [ ]:
import json, datetime as dt, numpy as np, pandas as pd, joblib
from sklearn.metrics import log_loss, brier_score_loss

TEST = DS[DS.block == "test"].reset_index(drop=True)
opened_file = CFG.path("logs") / "test_block_opened.json"
first_open = json.loads(opened_file.read_text()) if opened_file.exists() else None
if first_open is None:
    first_open = {"first_opened_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                  "test_dates": [str(TEST.date.min().date()), str(TEST.date.max().date())], "n": len(TEST)}
    opened_file.write_text(json.dumps(first_open, indent=2))
    print("ℹ️  Test block opened for the first time — recorded in logs/test_block_opened.json.")
else:
    print(f"⚠️  The test block was first opened on {first_open['first_opened_utc'][:16]} UTC. If you've changed the "
          "model since then because of what you saw here, these numbers are no longer fully independent.")

y_test = TEST.y.values
is_vct = TEST.tier.isin(TIER1).values
M, anti, feats = ART["models"], ART["antisym"], ART["features"]
sym = lambda f, X: 0.5 * (f(X) + 1 - f(mirror_cols(X, anti)))
PT = {
    "Elo": 1 / (1 + np.exp(-TEST.elo_logit.values)),
    "Elo (calibrated)": sym(lambda Z: M["Elo (calibrated)"].predict_proba(Z[["elo_logit"]])[:, 1], TEST[feats]),
    "Logistic Regression": sym(lambda Z: M["Logistic Regression"].predict_proba(Z[anti])[:, 1], TEST[feats]),
    "LightGBM": sym(lambda Z: M["LightGBM"].predict_proba(Z[feats])[:, 1], TEST[feats]),
}
PT["Ensemble"] = 0.5 * (PT["Logistic Regression"] + PT["LightGBM"])
FINAL_NAME = f"FINAL ({ART['chosen']} + calibration)"
PT[FINAL_NAME] = apply_calibration(base_predict(TEST[feats]), is_vct.astype(float), CAL_A, CAL_B)
print(f"✅ {len(TEST):,} test series ({TEST.date.min().date()} → {TEST.date.max().date()}), "
      f"{is_vct.sum():,} of them VCT")

### 8.2 Scores with 95% ranges

In [ ]:
def boot_ci(y, p, metric, n=CFG.bootstrap_samples, seed=CFG.seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), size=(n, len(y)))
    vals = np.array([metric(y[i], p[i]) for i in idx])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

LL = lambda y, p: log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1])
ACC = lambda y, p: float(np.mean((p > 0.5) == (y == 1)))
BRIER = lambda y, p: brier_score_loss(y, p)

def evaluate(mask, label):
    rows = []
    for name, p in PT.items():
        y, q = y_test[mask], p[mask]
        lo_a, hi_a = boot_ci(y, q, ACC)
        rows.append({"model": name, "log loss": LL(y, q), "brier": BRIER(y, q), "accuracy": ACC(y, q),
                     "accuracy 95%": f"{lo_a:.1%} – {hi_a:.1%}", "ECE": ece(q, y)})
    t = pd.DataFrame(rows).set_index("model")
    print(f"\n{label}  ({mask.sum():,} series)")
    print(t.to_string(formatters={"log loss": "{:.4f}".format, "brier": "{:.4f}".format,
                                  "accuracy": "{:.1%}".format, "ECE": "{:.3f}".format}))
    return t

ALL_T = evaluate(np.ones(len(TEST), bool), "ALL test series")
VCT_T = evaluate(is_vct, "VCT only (international + regional)")

### 8.3 Is it genuinely better than Elo?
A **paired bootstrap**: the same re-sampled matches are scored by both the final predictor and Elo, 1,000 times. If the whole 95% range of the difference is on the "better" side, the improvement is real; if it crosses zero, it could be luck.

In [ ]:
def paired(y, p_base, p_new, metric, n=CFG.bootstrap_samples, seed=CFG.seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), size=(n, len(y)))
    diffs = np.array([metric(y[i], p_new[i]) - metric(y[i], p_base[i]) for i in idx])
    return float(metric(y, p_new) - metric(y, p_base)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

VERDICT = {}
for label, mask in (("all", np.ones(len(TEST), bool)), ("VCT", is_vct)):
    y, pf, pe = y_test[mask], PT[FINAL_NAME][mask], PT["Elo"][mask]
    d_ll = paired(y, pe, pf, LL)
    d_acc = paired(y, pe, pf, ACC)
    better = d_ll[2] < 0
    worse = d_ll[1] > 0
    VERDICT[label] = {"n": int(mask.sum()), "logloss_diff": d_ll, "accuracy_diff": d_acc,
                      "verdict": "better" if better else ("worse" if worse else "not distinguishable")}
    print(f"{label:4s} ({mask.sum():5,} series): log loss {d_ll[0]:+.4f} [{d_ll[1]:+.4f}, {d_ll[2]:+.4f}]   "
          f"accuracy {d_acc[0]:+.1%} [{d_acc[1]:+.1%}, {d_acc[2]:+.1%}]   → "
          + {"better": "✅ genuinely better than Elo", "worse": "❌ worse than Elo",
             "not distinguishable": "➖ can't tell apart from Elo (difference could be luck)"}[VERDICT[label]["verdict"]])
print("   (log loss: negative = better; accuracy: positive = better)")

### 8.4 Calibration, breakdowns, and stability over time

In [ ]:
pf = PT[FINAL_NAME]
print("Reliability on the test block — when the model says X%, how often did the favourite win?")
print(reliability(pf, y_test).to_string(formatters={"predicted": "{:.0%}".format, "actual": "{:.0%}".format}))

print("\nBy tier and series length (final predictor vs Elo accuracy):")
br = []
for (tier, bo), g in TEST.assign(pf=pf, pe=PT["Elo"]).groupby(["tier", "best_of"]):
    if len(g) >= 20:
        br.append({"tier": tier, "best of": int(bo), "series": len(g), "final": ACC(g.y.values, g.pf.values),
                   "Elo": ACC(g.y.values, g.pe.values), "final log loss": LL(g.y.values, g.pf.values)})
print(pd.DataFrame(br).to_string(index=False, formatters={"final": "{:.1%}".format, "Elo": "{:.1%}".format,
                                                         "final log loss": "{:.4f}".format}))

print("\nMonth by month (log loss — lower is better):")
mon = TEST.assign(pf=pf, pe=PT["Elo"], month=TEST.date.dt.to_period("M")).groupby("month").apply(
    lambda g: pd.Series({"series": len(g), "final": LL(g.y.values, g.pf.values), "Elo": LL(g.y.values, g.pe.values)}),
    include_groups=False)
mon["final better"] = np.where(mon.final < mon.Elo, "yes", "no")
print(mon.to_string(formatters={"series": "{:.0f}".format, "final": "{:.4f}".format, "Elo": "{:.4f}".format}))
STABLE_SHARE = float((mon["final better"] == "yes").mean())

# Map model on the test block's maps
mt = MD[MD.block == "test"]
pm = sym_predict(lambda X: MAP_MODEL.predict_proba(X[MAP_ANTI])[:, 1], mt[MAP_ANTI], MAP_ANTI)
pb = 1 / (1 + 10 ** (-mt.d_map_rating.values / 400))
print(f"\nMap model on {len(mt):,} test maps: log loss {LL(mt.y.values, pm):.4f} vs map rating alone "
      f"{LL(mt.y.values, pb):.4f}; accuracy {ACC(mt.y.values, pm):.1%} vs {ACC(mt.y.values, pb):.1%}")

### 8.5 The verdict, in plain words

In [ ]:
fa, fv = ALL_T.loc[FINAL_NAME], VCT_T.loc[FINAL_NAME]
ea, ev = ALL_T.loc["Elo"], VCT_T.loc["Elo"]
words = {"better": "genuinely better than", "worse": "worse than", "not distinguishable": "not clearly different from"}
print(f"On {len(TEST):,} matches it had never seen ({TEST.date.min().date()} → {TEST.date.max().date()}):")
print(f"  • All matches: the final predictor picked the winner {fa['accuracy']:.1%} of the time "
      f"(95% range {fa['accuracy 95%']}) vs {ea['accuracy']:.1%} for Elo — {words[VERDICT['all']['verdict']]} Elo.")
print(f"  • VCT matches: {fv['accuracy']:.1%} ({fv['accuracy 95%']}) vs {ev['accuracy']:.1%} for Elo — "
      f"{words[VERDICT['VCT']['verdict']]} Elo ({VERDICT['VCT']['n']} matches).")
print(f"  • Calibration error (ECE): {fa['ECE']:.3f} — on average its percentages are off by about "
      f"{fa['ECE'] * 100:.0f} points.")
print(f"  • It beat Elo in {STABLE_SHARE:.0%} of test months.")

REPORT = {"test_dates": [str(TEST.date.min().date()), str(TEST.date.max().date())], "n_test": len(TEST),
          "n_vct": int(is_vct.sum()), "final_model": FINAL_NAME, "first_opened": first_open,
          "all": ALL_T.reset_index().to_dict("records"), "vct": VCT_T.reset_index().to_dict("records"),
          "verdict": VERDICT, "months_better_share": STABLE_SHARE}
(CFG.path("logs") / "layer8_report.json").write_text(json.dumps(REPORT, indent=2, default=str))
TEST.assign(p_final=pf, p_elo=PT["Elo"])[["match_id", "date", "tier", "best_of", "team1", "team2", "y",
                                         "p_final", "p_elo"]].to_parquet(CFG.path("logs") / "test_predictions.parquet", index=False)
log.info("Layer 8: %s", {k: v["verdict"] for k, v in VERDICT.items()})
print("\n🎉 Layer 8 complete — report saved to logs/layer8_report.json.")

---
# Layer 9 — The app

A small web app inside Colab: **pick two teams, see who's likely to win and why.**

| Tab | What it shows |
|---|---|
| **Predict** | Win chances, confidence and notes, the likely maps with each team's chance on them, and the top reasons for the prediction |
| **Upcoming matches** | Predictions for scheduled matches from the events in progress (saved by Layer 1 — re-run Layer 1 to refresh). Every prediction shown here is **logged**, so Layer 10 can later check it against the real result |
| **About** | How accurate the model was on the test block, and what it can't know |

Colab shows the app below the cell and also prints a **public link** (valid ~72 hours) you can open in another tab or on your phone. Stop it with the ■ button on the cell.

### 9.1 App logic

In [ ]:
import os, re, json, datetime as dt, numpy as np, pandas as pd

MATCH_TYPES = {"VCT league match (same region)": "regional", "VCT international (Masters / Champions)": "international",
               "Challengers": "challengers"}

# Team list: active in the last year, strongest first.
_cut = NOW - pd.Timedelta(days=365)
_active = [(ENGINE_LIVE.elo[k] + ENGINE_LIVE.offset(k), k) for k, d in ENGINE_LIVE.last_date.items() if d >= _cut]
TEAM_CHOICES = []
_seen = set()
for _, k in sorted(_active, reverse=True):
    nm = team_name(k)
    if nm not in _seen:
        TEAM_CHOICES.append(nm)
        _seen.add(nm)

def reasons_table(team1, team2, best_of, tier, top=6) -> pd.DataFrame:
    ex = explain_match(team1, team2, best_of, tier, top=100)
    push = ex.columns[2]
    is_elo = ex.feature.str.startswith("Elo")
    elo_total = ex.loc[is_elo, push].sum()
    rest = ex.loc[~is_elo, ["feature", push]]
    t = pd.concat([pd.DataFrame({"feature": ["overall strength (Elo)"], push: [elo_total]}), rest], ignore_index=True)
    t = t.reindex(t[push].abs().sort_values(ascending=False).index).head(top)
    n1, n2 = team_name(find_team(team1)), team_name(find_team(team2))
    # A logit push of v moves a 50/50 match by about 25·v percentage points.
    return pd.DataFrame({
        "factor": t.feature.values,
        "favours": [n1 if v > 0 else n2 for v in t[push].values],
        "rough effect": [f"≈ {abs(v) * 25:.0f} pts" if abs(v) * 25 >= 0.5 else "< 1 pt" for v in t[push].values]})

def _bar(n1, p1, n2, p2):
    return (f'<div style="font-family:sans-serif;margin:6px 0">'
            f'<div style="display:flex;height:34px;border-radius:8px;overflow:hidden;font-weight:600;color:white">'
            f'<div style="width:{p1:.1%};background:#2b6cb0;display:flex;align-items:center;padding-left:10px">{n1} {p1:.0%}</div>'
            f'<div style="width:{p2:.1%};background:#c53030;display:flex;align-items:center;justify-content:flex-end;'
            f'padding-right:10px">{p2:.0%} {n2}</div></div></div>')

def ui_predict(team1, team2, best_of, match_type):
    empty = pd.DataFrame()
    try:
        if not team1 or not team2:
            return "Pick two teams.", "", empty, empty
        tier = MATCH_TYPES.get(match_type, "regional")
        r = predict_match(team1, team2, best_of=int(best_of), tier=tier)
    except ValueError as e:
        return f"⚠️ {e}", "", empty, empty
    badge = "🟢 normal confidence" if r["confidence"] == "normal" else "🟠 low confidence"
    notes = "".join(f"\n- {n}" for n in r["notes"])
    head = (f"### {r['team1']} {r['p_team1']:.0%} — {r['p_team2']:.0%} {r['team2']}\n"
            f"Bo{r['best_of']} · {badge} · built map by map: {r['team1']} {r['breakdown']['series_from_maps']:.0%} · "
            f"data up to {r['data_up_to']}{notes}")
    maps = pd.DataFrame([{
        "map": m["map"], "chance it's played": f"{m['play_chance']:.0%}",
        f"{r['team1']} wins if they pick it": f"{m['t1_chance_if_t1_picks']:.0%}",
        f"… if {r['team2']} picks it": f"{m['t1_chance_if_t2_picks']:.0%}",
        "… as the decider": f"{m['t1_chance_as_decider']:.0%}"} for m in r["breakdown"]["maps"]])
    return head, _bar(r["team1"], r["p_team1"], r["team2"], r["p_team2"]), maps, \
        reasons_table(team1, team2, int(best_of), tier)

# ---- upcoming matches (from Layer 1's saved event files) ----
PRED_LOG = CFG.path("logs") / "prediction_log.csv"
_MON = "january|february|march|april|may|june|july|august|september|october|november|december"

def _date(text, unix_ts=""):
    """'Sat, October 25, 2026' → date (the text is trusted first, as in Layers 1–2)."""
    m = re.search(rf"({_MON})\s+(\d{{1,2}}),?\s+(20\d\d)", (text or "").lower())
    try:
        if m:
            return dt.date(int(m.group(3)), _MON.split("|").index(m.group(1)) + 1, int(m.group(2)))
        if unix_ts and re.match(r"^\d{4}-\d{2}-\d{2}", str(unix_ts)):
            return dt.date.fromisoformat(str(unix_ts)[:10])
    except ValueError:
        pass
    return None

def upcoming_matches(limit=60) -> pd.DataFrame:
    events = json.loads((CFG.path("events") / "events.json").read_text()).get("events", {}) \
        if (CFG.path("events") / "events.json").exists() else {}
    today = dt.datetime.now(dt.timezone.utc).date()
    rows = []
    for f in sorted(CFG.path("event_matches").glob("*.json")):
        saved = json.loads(f.read_text())
        title = events.get(saved["event_id"], {}).get("title", "")
        tier = classify_event(title)
        if tier is None:
            continue
        for m in saved["matches"]:
            if m.get("status", "").strip().lower() == "completed":
                continue
            n1, n2 = m["team1"].get("name", ""), m["team2"].get("name", "")
            d = _date(m.get("date", ""), m.get("unix_timestamp", ""))
            if "TBD" in (n1.upper(), n2.upper()) or not n1 or not n2 or d is None or d < today - dt.timedelta(days=1):
                continue
            try:
                k1, k2 = find_team(n1), find_team(n2)
            except ValueError:
                continue                                   # a team we have no record of
            series = str(m.get("event_series", ""))
            bo = 5 if re.search(r"grand final|lower final", series, re.I) else 3
            r = predict_match(n1, n2, best_of=bo, tier=tier, with_maps=False)
            rows.append({"date": str(d), "event": title, "round": series, "team 1": r["team1"],
                         "chance 1": f"{r['p_team1']:.0%}", "chance 2": f"{r['p_team2']:.0%}", "team 2": r["team2"],
                         "best of": bo, "confidence": r["confidence"],
                         "_match_id": str(m.get("match_id", "")), "_p1": r["p_team1"]})
    t = pd.DataFrame(rows)
    if t.empty:
        return t
    t = t.sort_values(["date", "event"]).head(limit).reset_index(drop=True)
    log_predictions(t)
    return t

def log_predictions(t: pd.DataFrame):
    """Remember each upcoming match's prediction once (the first time it's shown) for Layer 10."""
    old = pd.read_csv(PRED_LOG, dtype={"match_id": str}) if PRED_LOG.exists() else pd.DataFrame(columns=["match_id"])
    new = t[~t._match_id.isin(set(old.match_id))]
    if new.empty:
        return
    add = pd.DataFrame({"match_id": new._match_id, "predicted_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                        "match_date": new.date, "event": new.event, "team1": new["team 1"], "team2": new["team 2"],
                        "best_of": new["best of"], "p_team1": new._p1.round(4), "model": ART["chosen"],
                        "data_up_to": str((NOW - pd.Timedelta(days=1)).date())})
    pd.concat([old, add], ignore_index=True).to_csv(PRED_LOG, index=False)

def ui_upcoming():
    t = upcoming_matches()
    if t.empty:
        return pd.DataFrame({"info": ["No scheduled matches with known teams in the saved event files. "
                                      "Re-run Layer 1 to refresh events in progress."]})
    return t.drop(columns=["_match_id", "_p1"])

def about_text() -> str:
    rep_file = CFG.path("logs") / "layer8_report.json"
    if not rep_file.exists():
        return "Run Layer 8 first to see the model's test scores."
    rep = json.loads(rep_file.read_text())
    fin_all = next(r for r in rep["all"] if r["model"].startswith("FINAL"))
    fin_vct = next(r for r in rep["vct"] if r["model"].startswith("FINAL"))
    elo_all = next(r for r in rep["all"] if r["model"] == "Elo")
    elo_vct = next(r for r in rep["vct"] if r["model"] == "Elo")
    words = {"better": "genuinely better than", "worse": "worse than", "not distinguishable": "not clearly different from"}
    return f"""### How good is this?
Tested once on **{rep['n_test']:,} matches it had never seen** ({rep['test_dates'][0]} → {rep['test_dates'][1]}):

| | This model | Elo baseline |
|---|---|---|
| All matches — picked the winner | **{fin_all['accuracy']:.1%}** ({fin_all['accuracy 95%']}) | {elo_all['accuracy']:.1%} |
| VCT matches ({rep['n_vct']}) | **{fin_vct['accuracy']:.1%}** ({fin_vct['accuracy 95%']}) | {elo_vct['accuracy']:.1%} |
| Average error of the percentages | {fin_all['ECE'] * 100:.1f} points | |

Verdict vs Elo: all matches — {words[rep['verdict']['all']['verdict']]}; VCT — {words[rep['verdict']['VCT']['verdict']]}.

### What it can't know
- Last-minute roster changes, stand-ins, illness — it rates the lineup from each team's latest recorded match.
- Patch and agent meta shifts beyond a team's record on the current patch.
- Anything after **{(NOW - pd.Timedelta(days=1)).date()}** — re-run the notebook to update.

Pro Valorant is close and random: roughly 6 in 10 is a solid result. **Please don't use this for betting.**
"""
print(f"✅ App logic ready — {len(TEAM_CHOICES):,} teams active in the last year.")

### 9.2 Launch the app

In [ ]:
import gradio as gr

def build_app():
    with gr.Blocks() as app:
        gr.Markdown("# Valorant match predictor\nPick two teams. Win chances come from the model tested in Layer 8.")
        with gr.Tab("Predict"):
            with gr.Row():
                t1 = gr.Dropdown(TEAM_CHOICES, value=TEAM_CHOICES[0] if TEAM_CHOICES else None, label="Team 1",
                                 filterable=True)
                t2 = gr.Dropdown(TEAM_CHOICES, value=TEAM_CHOICES[1] if len(TEAM_CHOICES) > 1 else None,
                                 label="Team 2", filterable=True)
            with gr.Row():
                bo = gr.Radio(["1", "3", "5"], value="3", label="Best of")
                mt = gr.Radio(list(MATCH_TYPES), value=list(MATCH_TYPES)[0], label="Match type")
            go = gr.Button("Predict", variant="primary")
            head, bar = gr.Markdown(), gr.HTML()
            gr.Markdown("**Likely maps** — chance each map is played, and Team 1's chance on it")
            maps = gr.Dataframe(interactive=False, wrap=True)
            gr.Markdown("**Why** — the biggest factors, read from the Logistic Regression model (a close guide, not exact "
                        "for the final model)")
            why = gr.Dataframe(interactive=False)
            go.click(ui_predict, [t1, t2, bo, mt], [head, bar, maps, why])
        with gr.Tab("Upcoming matches"):
            gr.Markdown("Scheduled matches from events in progress. Best-of is a guess (Bo5 for grand/lower finals, "
                        "otherwise Bo3). Each prediction is logged the first time it's shown, for Layer 10.")
            refresh = gr.Button("Load / refresh predictions")
            up = gr.Dataframe(interactive=False, wrap=True)
            refresh.click(ui_upcoming, [], [up])
        with gr.Tab("About"):
            gr.Markdown(about_text())
    return app

APP = build_app()
if os.environ.get("VALPRED_NO_LAUNCH") != "1":
    APP.launch(share=IS_COLAB, debug=False)